## 1. Setup


In [ ]:

import base64
import csv
import glob
import io
import json
import os
import re
import time
import xml.etree.ElementTree as ET
from typing import Optional

import boto3
import numpy as np
import pandas as pd
import requests
from anthropic import Anthropic
from openai import OpenAI


In [ ]:

# Local paths. Change data_path for your machine; everything else is derived from it.
data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover'
ctd_data_path = data_path + '/CTD_data'
pubtator_data_path = data_path + '/pubtator_data'
input_data_path = data_path + '/BedrockInput'
output_data_path = data_path + '/BedrockOutput'
figures_path = data_path + '/BedrockFigures'
supplement_path = data_path + '/BedrockSupplement'

## 2. Data: CTD and PubTator reference set


### PubTator reference set (papers sent to extraction)


In [ ]:
pm_pmc_dict = {
    '9118903': '1469763',
    '15154616': '2275409',
    '15811834': '1278483',
    '15953866': '2782200',
    '16393664': '1332662',
    '17118201': '1676006',
    '17183729': '1762369',
    '17524151': '1890552',
    '18280460': '2279149',
    '18299717': '2243244',
    '19114014': '2631511',
    '19617202': '3136680',
    '19654922': '2717139',
    '20801540': '2987498',
    '21294877': '3042928',
    '22110480': '3205718',
    '22112310': '3279445',
    '22228119': '3349910',
    '22562770': '3440069',
    '22571318': '3485091',
    '22747577': '3489685',
    '22808131': '3392256',
    '23006841': '3488501',
    '23349674': '3551951',
    '23568217': '3620667',
    '24039778': '3769399',
    '24403256': '3892387',
    '24413757': '3907846',
    '25361045': '4245613',
    '25859307': '4387225',
    '26039251': '4454644',
    '26492236': '4632778',
    '26714306': '4695093',
    '27153807': '4859981',
    '27397436': '5298042',
    '27517156': '5295416',
    '27736846': '5129817',
    '28637469': '5480112',
    '29045497': '5646803',
    '29380240': '6060758',
    '30104959': '6077207',
    '30114287': '6095593',
    '31064381': '6503545',
    '31378678': '6706627',
    '32298299': '7161975',
    '34908893': '8640676',
    '36638633': '10016736',
    '37047793': '10095583',
    '25100957': '4104784',
    '19889203': '3003513'
}
pmids = list(pm_pmc_dict.keys())

url = "https://www.ncbi.nlm.nih.gov/research/pubtator3-api/publications/export/biocjson?pmids="

url += ",".join(str(p) for p in pmids) + "&full=true"
print(url)
# Save to a file only once, uncomment for replication
# response = requests.get(url)

# response.raise_for_status()

# data = response.json()

# with open(f"{pubtator_data_path}/{pubtatorFileName}", "w") as f:
#     json.dump(data, f, indent=4)

# print(f"Saved {pubtatorFileName}")


## 3. Figures, supplements, and PubTator processing


### Figures (defines the shared PMC download helpers used by the supplement cell)


In [ ]:

BUCKET = "https://pmc-oa-opendata.s3.amazonaws.com"
S3NS = "{http://s3.amazonaws.com/doc/2006-03-01/}"
XLINK_HREF = "{http://www.w3.org/1999/xlink}href"
IMG_EXT = {".jpg", ".jpeg", ".png", ".gif", ".tif", ".tiff"}
HEADERS = {"User-Agent": "research-script/1.0"}

def _get(url, **kw):
    for attempt in range(3):
        r = requests.get(url, headers=HEADERS, timeout=120, **kw)
        if r.status_code in (429, 500, 502, 503):
            time.sleep(2 ** attempt); continue
        r.raise_for_status()
        return r
    r.raise_for_status()

def _clean(elem):
    return re.sub(r"\s+", " ", "".join(elem.itertext())).strip() if elem is not None else ""

def _norm_pmcid(pmcid):
    return "PMC" + re.sub(r"\D", "", str(pmcid))

def _list_keys(prefix, delimiter=None):
    params = {"list-type": "2", "prefix": prefix}
    if delimiter:
        params["delimiter"] = delimiter
    root = ET.fromstring(_get(BUCKET + "/", params=params).content)
    keys = [k.text for k in root.iter(S3NS + "Key")]
    prefixes = [p.text for p in root.iter(S3NS + "Prefix") if p.text != prefix]
    return keys, prefixes

def _pick_version(pmcid):
    """Return e.g. 'PMC3488501.1'. Prefers non-manuscript versions."""
    _, prefixes = _list_keys(f"{pmcid}.", delimiter="/")
    versions = sorted(p.rstrip("/") for p in prefixes)
    if not versions:
        raise RuntimeError(f"{pmcid}: not in PMC cloud dataset (not OA / no TDM license)")
    for v in versions:
        meta = _get(f"{BUCKET}/metadata/{v}.json").json()
        if str(meta.get("is_manuscript", "")).lower() not in ("yes", "true"):
            return v
    return versions[0]

def download_pmc_figures_and_tables(pmcid, pmid, outdir=figures_path):
    os.makedirs(outdir, exist_ok=True)
    pmcid = _norm_pmcid(pmcid)
    csv_path = os.path.join(outdir, f"{pmcid}_figures_metadata.csv")

    ver = _pick_version(pmcid)
    keys, _ = _list_keys(f"{ver}/")
    by_stem = {}
    for k in keys:
        stem, ext = os.path.splitext(os.path.basename(k))
        if ext.lower() in IMG_EXT:
            by_stem.setdefault(stem, k)

    root = ET.fromstring(_get(f"{BUCKET}/{ver}/{ver}.xml").content)

    count = 1
    with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=[
            "pmid", "number", "caption", "filename", "orginal_filename", "type"])
        writer.writeheader()

        for elem in root.iter():
            if elem.tag not in ("fig", "table-wrap"):
                continue
            kind = "figure" if elem.tag == "fig" else "table"
            caption = _clean(elem.find("caption"))

            for j, g in enumerate(elem.iter("graphic"), 1):
                href = g.get(XLINK_HREF) or ""
                src = by_stem.get(os.path.splitext(href)[0])
                if not src:
                    continue
                base = os.path.basename(src)
                filename = f"{pmid}_{kind}{count}_img{j}_{base}"
                with open(os.path.join(outdir, filename), "wb") as f:
                    f.write(_get(f"{BUCKET}/{src}").content)
                print(f"Downloaded {kind}: {filename}")
                writer.writerow({"pmid": pmid, "number": count, "caption": caption,
                                 "filename": filename, "orginal_filename": base,
                                 "type": kind})
                count += 1

    print(f"✅ {pmcid} ({ver}): {count-1} items")

def get_figuredata(pmcid, pmid, figures_path):
    download_pmc_figures_and_tables(pmcid, pmid, figures_path)


In [ ]:
PREFIX_LEN = 15

def normalize_table_caption(text):
    if not text or pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"[^a-z0-9]", "", text)
    return text

def match_pubtator_fig_annotations(csv_dir, pubtator_file):
    """
    Match PMC FIGURES by filename and TABLES by caption prefix.
    Does NOT add helper columns to the output DataFrame.
    """

    # 1️⃣ Load CSVs
    all_csvs = [os.path.join(csv_dir, f) for f in os.listdir(csv_dir) if f.endswith(".csv")]
    df = pd.concat([pd.read_csv(f) for f in all_csvs], ignore_index=True)
    print(f"Loaded {len(df)} rows from {len(all_csvs)} CSVs")

    # 2️⃣ Load PubTator JSON
    with open(pubtator_file, "r", encoding="utf-8") as f:
        pubtator_data = json.load(f)

    # 3️⃣ Build mappings
    fig_map = {}    # pmid -> filename -> annotations
    table_map = {}  # pmid -> caption_prefix -> annotations

    for entry in pubtator_data.get("PubTator3", []):
        pmid = entry["_id"].split("|")[0]

        fig_map.setdefault(pmid, {})
        table_map.setdefault(pmid, {})

        for passage in entry.get("passages", []):
            infons = passage.get("infons", {})
            section_type = infons.get("section_type", "")
            ptype = infons.get("type", "").lower()
            annotations = passage.get("annotations", [])

            # FIGURES
            if section_type == "FIG" and ptype in {"fig_caption", "fig_title_caption"}:
                file_name = infons.get("file")
                if file_name:
                    fig_map[pmid][file_name] = annotations

            # TABLES
            elif section_type == "TABLE" and ptype in {"table_caption", "table_title_caption"}:
                caption_text = passage.get("text", "")
                norm = normalize_table_caption(caption_text)
                if norm:
                    table_map[pmid][norm[:PREFIX_LEN]] = annotations

    # 4️⃣ Extract entities
    def extract_entities(row):
        pmid = str(row["pmid"])
        orginal_filename = row.get("orginal_filename", "")
        caption = row.get("caption", "")

        genes = {}
        chemicals = {}

        annotations = []

        # FIG match
        if pmid in fig_map and orginal_filename in fig_map[pmid]:
            annotations = fig_map[pmid][orginal_filename]

        # TABLE match
        else:
            cap_norm = normalize_table_caption(caption)
            cap_prefix = cap_norm[:PREFIX_LEN]
            if pmid in table_map and cap_prefix in table_map[pmid]:
                annotations = table_map[pmid][cap_prefix]

        for ann in annotations:
            infons = ann.get("infons", {})
            if not infons.get("valid", True):
                continue

            ann_type = infons.get("type", "").lower()
            name = ann.get("text") or infons.get("name")
            identifier = (
                infons.get("normalized_id")
                or infons.get("identifier")
                or infons.get("accession")
            )

            if not name or not identifier:
                continue

            identifier = str(identifier).upper()

            if ann_type == "gene":
                genes[name] = identifier
            elif ann_type == "chemical":
                chemicals[name] = identifier

        genes_str = "\n".join(f"{k}: {v}" for k, v in genes.items()) if genes else None
        chemicals_str = "\n".join(f"{k}: {v}" for k, v in chemicals.items()) if chemicals else None

        return pd.Series([genes_str, chemicals_str])

    df[["gene_map", "chemical_map"]] = df.apply(extract_entities, axis=1)

    print("✅ FIG (filename) and TABLE (caption prefix) annotations matched successfully.")
    return df


### Supplements


In [ ]:

# Supplement constants. BUCKET, S3NS, XLINK_HREF, HEADERS, IMG_EXT and the
# download helpers (_get, _clean, _norm_pmcid, _list_keys, _pick_version)
# come from the figures cell above.
TABULAR_EXT = {".xls", ".xlsx", ".xlsm", ".csv", ".tsv"}
GENE_RE = re.compile(r"gene", re.IGNORECASE)
FC_RE = re.compile(r"fold[\s\-]?change", re.IGNORECASE)
HEADER_SCAN_ROWS = 15
NUMERIC_FRAC = 0.9  # fold-change col must be >=90% numeric

# ---------------- supplement filtering helpers ----------------
def _check_engines():
    missing = []
    for mod in ("xlrd", "openpyxl"):
        try:
            __import__(mod)
        except ImportError:
            missing.append(mod)
    if missing:
        raise ImportError(f"Missing Excel readers: pip install {' '.join(missing)}")

def _read_raw_sheets(path):
    """{sheet_name: raw DataFrame, header=None}. csv/tsv -> one pseudo-sheet."""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".csv", ".tsv"):
        return {"sheet1": pd.read_csv(path, sep=None, engine="python", header=None)}
    return pd.read_excel(path, sheet_name=None, header=None)

def _promote_header(raw):
    """First row (top HEADER_SCAN_ROWS) with a gene/fold-change cell becomes the header."""
    for i in range(min(HEADER_SCAN_ROWS, len(raw))):
        cells = [str(v) for v in raw.iloc[i].tolist() if pd.notna(v)]
        if any(GENE_RE.search(c) or FC_RE.search(c) for c in cells):
            names, seen = [], {}
            for j, v in enumerate(raw.iloc[i].tolist()):
                n = str(v).strip() if pd.notna(v) and str(v).strip() else f"col{j}"
                seen[n] = seen.get(n, 0) + 1
                names.append(n if seen[n] == 1 else f"{n}.{seen[n]-1}")
            df = raw.iloc[i + 1:].copy()
            df.columns = names
            return df.reset_index(drop=True)
    return None

def filter_supp(df):
    cols = [str(c) for c in df.columns]
    df = df.copy()
    df.columns = cols

    gene_cols = [c for c in cols if GENE_RE.search(c)]
    fc_cols = [c for c in cols if FC_RE.search(c)]

    if len(fc_cols) > 1:  # multiple FC cols -> keep (nearly) all-numeric ones
        def mostly_numeric(c):
            s = df[c].dropna()
            return (not s.empty) and pd.to_numeric(s, errors="coerce").notna().mean() >= NUMERIC_FRAC
        fc_cols = [c for c in fc_cols if mostly_numeric(c)]

    keep = list(dict.fromkeys(gene_cols + fc_cols))
    out = df.loc[:, keep].copy()
    for c in fc_cols:  # stray "NA"/"inf" strings -> NaN
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def _filter_and_replace(path):
    """
    Filters every sheet of the file at `path` and overwrites it with the result.
    - csv/tsv: rewritten in place.
    - xls/xlsx/xlsm: rewritten as .xlsx (xls can't be written) with only the
      sheets that matched; the original is removed.
    - If nothing matches (or the file can't be read), the original is left untouched.
    Returns (final_filename, kept_sheet_names, was_filtered).
    """
    orig_name = os.path.basename(path)
    try:
        raw_sheets = _read_raw_sheets(path)
    except Exception as e:
        print(f"❌ Could not read {orig_name}: {e}")
        return orig_name, [], False

    kept = {}
    for name, raw in raw_sheets.items():
        df = _promote_header(raw)
        if df is None:
            print(f"skip {orig_name}[{name}]: no gene/fold-change header in first {HEADER_SCAN_ROWS} rows")
            continue
        filtered = filter_supp(df)
        if filtered.shape[1] == 0:
            print(f"skip {orig_name}[{name}]: no matching columns after filter")
            continue
        kept[name] = filtered

    if not kept:
        print(f"⚠️ {orig_name}: nothing matched, original kept")
        return orig_name, [], False

    ext = os.path.splitext(path)[1].lower()
    if ext in (".csv", ".tsv"):
        next(iter(kept.values())).to_csv(path, index=False, sep="\t" if ext == ".tsv" else ",")
        return orig_name, list(kept), True

    out_path = os.path.splitext(path)[0] + ".xlsx"
    tmp_path = out_path + ".tmp"
    used = set()
    with pd.ExcelWriter(tmp_path, engine="openpyxl") as xw:
        for name, df in kept.items():
            sname = re.sub(r"[\[\]:*?/\\]", "_", str(name))[:31] or "sheet"
            k = 1
            while sname.lower() in used:  # Excel sheet names must be unique (case-insens.)
                k += 1
                sname = f"{sname[:28]}_{k}"
            used.add(sname.lower())
            df.to_excel(xw, sheet_name=sname, index=False)
    os.replace(tmp_path, out_path)
    if out_path != path:
        os.remove(path)
    return os.path.basename(out_path), list(kept), True

# ---------------- supplements: download ----------------
def download_pmc_table_supplements(pmcid, pmid, outdir):
    """
    Downloads xls/xlsx/csv/tsv supplements as {pmid}_{original_name}.
    Writes {pmcid}_supplements_metadata.csv. No filtering here.
    """
    os.makedirs(outdir, exist_ok=True)
    pmcid = _norm_pmcid(pmcid)
    csv_path = os.path.join(outdir, f"{pmcid}_supplements_metadata.csv")

    ver = _pick_version(pmcid)
    keys, _ = _list_keys(f"{ver}/")
    files = {os.path.basename(k): k for k in keys
             if os.path.splitext(k)[1].lower() in TABULAR_EXT}
    if not files:
        print(f"{pmcid}: no tabular supplements in bucket")
        return

    root = ET.fromstring(_get(f"{BUCKET}/{ver}/{ver}.xml").content)
    described = {}  # file basename -> (label, caption)
    for sm in root.iter("supplementary-material"):
        label, caption = _clean(sm.find("label")), _clean(sm.find("caption"))
        for child in sm.iter():
            href = child.get(XLINK_HREF)
            if href:
                described.setdefault(os.path.basename(href), (label, caption))

    rows = []
    for count, (base, key) in enumerate(sorted(files.items()), 1):
        label, caption = described.get(base, ("", ""))
        filename = f"{pmid}_{base}"
        with open(os.path.join(outdir, filename), "wb") as f:
            f.write(_get(f"{BUCKET}/{key}").content)
        print(f"Downloaded supplement: {filename}")
        rows.append({"pmid": pmid, "number": count, "label": label, "caption": caption,
                     "filename": filename, "orginal_filename": base,
                     "type": "supplement_table", "sheets": "", "filtered": False})

    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f"✅ {pmcid} ({ver}): {len(rows)} supplement files")

# ---------------- supplements: filter (replaces files in place) ----------------
def filter_supplements(pmcid, pmid, supplement_path):
    """
    Filters each supplement listed in {pmcid}_supplements_metadata.csv and overwrites
    the file with the filtered result (.xls becomes .xlsx). Updates the metadata CSV
    (filename, sheets, filtered).
    """
    _check_engines()
    pmcid = _norm_pmcid(pmcid)
    meta_path = os.path.join(supplement_path, f"{pmcid}_supplements_metadata.csv")
    if not os.path.exists(meta_path):
        print(f"{pmcid}: no supplement metadata (nothing to filter)")
        return []
    meta = pd.read_csv(meta_path).fillna("")
    meta["sheets"] = meta["sheets"].astype(object)
    meta["filtered"] = meta["filtered"].astype(object)

    for i, row in meta.iterrows():
        path = os.path.join(supplement_path, row["filename"])
        if not os.path.exists(path):
            print(f"❌ Missing file: {row['filename']}")
            continue
        final_name, sheets, was_filtered = _filter_and_replace(path)
        meta.at[i, "filename"] = final_name
        meta.at[i, "sheets"] = "|".join(map(str, sheets))
        meta.at[i, "filtered"] = was_filtered
        print(f"Supplement saved: {final_name} (filtered={was_filtered}, sheets={len(sheets)})")

    meta.to_csv(meta_path, index=False)
    print(f"✅ {pmcid}: filtered {int(meta['filtered'].astype(bool).sum())}/{len(meta)} supplement files")
    return meta.to_dict("records")

def get_supplement_data(pmcid, pmid, supplement_path):
    download_pmc_table_supplements(pmcid, pmid, supplement_path)
    filter_supplements(pmcid, pmid, supplement_path)


### PubTator processing


In [ ]:
def process_pubtator_json(json_path, download_missing_tables=True):
    """Process PubTator JSON and return a DataFrame with deduplicated chem/gene maps."""
    with open(json_path, "r") as f:
        data = json.load(f)

    rows = []
    counter = 0

    for entry in data["PubTator3"]:
        pmid, pmcid = entry["_id"].split("|")

        # Step 1: Group TABLE passages
        group_passages = {}
        for passage in entry.get("passages", []):
            section_type = passage["infons"].get("section_type", "")
            ptype = passage["infons"].get("type", "")
            pid = passage["infons"].get("id", "")
            text = passage.get("text", "")
            annotations = passage.get("annotations", [])
            xml_data = passage["infons"].get("xml", "") if section_type=="TABLE" and ptype=="table" else ""

            if section_type in ["TABLE"] and pid:
                if pid not in group_passages:
                    group_passages[pid] = {"caption": "", "body": "", "annotations": [], "type": section_type, "xml": ""}

                if ptype.endswith("caption"):
                    group_passages[pid]["caption"] = text
                elif ptype in ["table", "fig"]:
                    group_passages[pid]["body"] = text
                    if xml_data:
                        group_passages[pid]["xml"] = xml_data

                group_passages[pid]["annotations"].extend(annotations)

        # Step 2: Process grouped TABLE
        for pid, pdata in group_passages.items():
            chem_map = {}
            gene_map = {}

            for ann in pdata["annotations"]:
                info = ann["infons"]
                ann_type = info.get("type", "").lower()
                ann_text = ann.get("text")
                ann_id = info.get("normalized_id") or ("E0000" if ann_type=="chemical" else "0000")
                ann_id_str = str(ann_id)
                ann_id_norm = ann_id_str.upper()

                if not ann_text:
                    continue

                # Deduplicate based on normalized ID (case-insensitive)
                if ann_type == "chemical":
                    if ann_id_norm not in {str(v).upper() for v in chem_map.values()}:
                        chem_map[ann_text] = ann_id
                elif ann_type == "gene":
                    if ann_id_norm not in {str(v).upper() for v in gene_map.values()}:
                        gene_map[ann_text] = ann_id

            if chem_map or gene_map:
                row = {
                    "id": f"{pubtatorFileName}{counter}",
                    "pmid": pmid,
                    "pmcid": pmcid,
                    "section_type": pdata["type"],
                    "text": "",
                    "chem_map": json.dumps(chem_map),
                    "gene_map": json.dumps(gene_map)
                }

                if pdata["type"] == "TABLE":
                    row["table_caption"] = pdata["caption"]
                    row["table_text"] = pdata["body"]
                    row["table_xml"] = pdata["xml"]

                    # # download missing tables as figure
                    # if download_missing_tables and not row["table_text"]:
                    #     table_file = download_table_as_figure(pmcid, pid)
                    #     row["table_text"] = f"Downloaded as figure: {table_file}" if table_file else "none"

                # else:  # FIG
                    # row["fig_caption"] = pdata["caption"]
                    # row["fig_text"] = pdata["body"]

                rows.append(row)
                counter += 1

        # Step 3: Process normal passages (non-TABLE/)
        for passage in entry.get("passages", []):
            section_type = passage["infons"].get("section_type", "")
            if section_type in ["TABLE", "FIG"]:
                continue

            text = passage.get("text", "")
            annotations = passage.get("annotations", [])

            chem_map = {}
            gene_map = {}

            for ann in annotations:
                info = ann["infons"]
                ann_type = info.get("type", "").lower()
                ann_text = ann.get("text")
                ann_id = info.get("normalized_id") or ("E0000" if ann_type=="chemical" else "0000")
                ann_id_str = str(ann_id)
                ann_id_norm = ann_id_str.upper()

                if not ann_text:
                    continue

                if ann_type == "chemical":
                    if ann_id_norm not in {str(v).upper() for v in chem_map.values()}:
                        chem_map[ann_text] = ann_id
                elif ann_type == "gene":
                    if ann_id_norm not in {str(v).upper() for v in gene_map.values()}:
                        gene_map[ann_text] = ann_id

            if chem_map and gene_map:
                rows.append({
                    "id": f"{pubtatorFileName}{counter}",
                    "pmid": pmid,
                    "pmcid": pmcid,
                    "section_type": section_type,
                    "text": text,
                    "table_caption": "",
                    "table_text": "",
                    "table_xml": "",
                    "chem_map": json.dumps(chem_map),
                    "gene_map": json.dumps(gene_map)
                })
                counter += 1

    df = pd.DataFrame(rows)
    df = df.sort_values(by="pmid").reset_index(drop=True)
    return df


## 4. Run figures, supplements, and PubTator processing


In [ ]:
# download figures
for pmid, pmcid in pm_pmc_dict.items():
    try:
        print(f"Processing PMID: {pmid}, PMCID: {pmcid}")
        get_figuredata(pmcid, pmid, figures_path)
    except Exception as e:
        print(f"❌ Failed for PMID {pmid}, PMCID {pmcid}: {e}")

df = match_pubtator_fig_annotations(figures_path, f'{pubtator_data_path}/{pubtatorFileName}')
df.to_csv(f'{figures_path}/figures.csv')


In [ ]:
supp_failed = []
for pmid, pmcid in pm_pmc_dict.items():
    try:
        print(f"Processing supplements, PMID: {pmid}, PMCID: {pmcid}")
        get_supplement_data(pmcid, pmid, supplement_path)
    except Exception as e:
        print(f"❌ Supplements failed for PMID {pmid}, PMCID {pmcid}: {e}")
        supp_failed.append((pmid, pmcid, str(e)))


In [ ]:
figs = pd.read_csv(f'{figures_path}/figures.csv').drop(columns=['Unnamed: 0'], axis=1)
figs


In [ ]:
text = process_pubtator_json(f"{pubtator_data_path}/{pubtatorFileName}")
text


## 5. Prompts 

In [ ]:
def fig_prompt(caption):
    systemPrompt = (
        "You are an AI assistant specialized in extracting relationships between chemicals "
        "and genes from images in scientific papers. Be precise, helpful, and honest. "
        "If a relationship is unclear or not present, state that clearly. Only use data from "
        "the image and caption; do not assume anything.\n"
        "Pay close attention to visual elements such as arrows, color coding, bar graphs, "
        "line graphs, labels, annotations, fold-change indicators, and highlighted elements, "
        "as these often indicate key relationships.\n"
        "Do not extract relationships from negative or non-significant results."
    )
    mainPrompt = f"""### Extract Specific Relationships Between Chemicals and Genes from image

#### Image Caption:
{caption}

---

### Provided Relationship List
The following relationships are predefined and must be used to describe chemical-gene interactions:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Instructions
For each extracted relationship, generate a CSV object with the following fields:
- `chemicalName`: Name of the chemical.
- `relationship`: This must be chosen from the provided list of relationships. The relationship may be prefixed by `increase`, `decrease`, or `affect`.
- `geneName`: Gene name if provided, otherwise leave blank. Gene names are longer than gene symbol.
- `geneSymbol`: Gene symbol if provided, otherwise leave blank. Gene symbols are typically all in uppercase.

#### Guidelines:
1. Extract specific relationships between these chemicals and genes using only the predefined terms from the provided relationship list. Do not use any terms or descriptions of interactions from the source text unless they exactly match terms from the list.
2. Provide only the CSV output—no additional explanations or comments.
3. Ensure that your CSV output is well-formed and follows CSV format.
4. Recheck to ensure the relationship is in the provided relationship list.
5. Use your best judgement to extract chemical-gene relationships, in some cases there might not be any. If there are no relationships, leave the csv empty.

---

### Example Output

#### Example Image Caption:
Gene regulation by Genistein.

#### Example Image:
Assume the image is passed in and you parsed the image internally.

#### Example CSV Output:
chemicalName,relationship,geneName,geneSymbol
"Genistein","Decreases expression","-","DEF2"
"Genistein","Increases expression","-","ABC1"
"Genistein","Decreases activity","tyrosine 3-monooxygenase","-"
"""
    return systemPrompt, mainPrompt


In [ ]:
def fig_table_prompt(caption):
    """
    Generates a system prompt and main prompt for a table in a figure.
    Returns systemPrompt, mainPrompt.
    """
    systemPrompt = (
        "You are an AI assistant specialized in extracting relationships between chemicals "
        "and genes from XML tables in scientific papers. Be precise, helpful, and honest. "
        "If a relationship is unclear or not present, state that clearly. Only use the data "
        "from the provided table; do not assume anything.\n"
        "Do not extract relationships from negative or non-significant results."
    )
    mainPrompt = f"""### Extract Specific Relationships Between Chemicals and Genes from Table

#### Table Caption:
{caption}

---

### Provided Relationship List
The following relationships are predefined and must be used to describe chemical-gene interactions:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Instructions
For each extracted relationship, generate a CSV object with the following fields:
- `chemicalName`: Name of the chemical.
- `relationship`: This must be chosen from the provided list of relationships. The relationship may be prefixed by `increase`, `decrease`, or `affect`.
- `geneName`: Gene name if provided, otherwise leave blank. Gene names are longer than gene symbol.
- `geneSymbol`: Gene symbol if provided, otherwise leave blank. Gene symbols are typically all in uppercase.

#### Guidelines:
1. Extract specific relationships between these chemicals and genes using only the predefined terms from the provided relationship list. Do not use any terms or descriptions of interactions from the source text unless they exactly match terms from the list.
2. Provide only the CSV output—no additional explanations or comments.
3. Ensure that your CSV output is well-formed and follows CSV format.
4. Recheck to ensure the relationship is in the provided relationship list.
5. In tabular data, relationships of interest are typically increase expression or decrease expression unless otherwise specified.
6. There may not be any relationships; in this case leave the output blank.
7. Do not extract relationships from negative or non-significant results.

---

### Example Output

#### Example Table Caption:
Gene regulation by Genistein. Table contains differentially expressed genes (p<0.05).

#### Example Table:
Assume the Table is passed in as an image, and you parsed the image internally.

#### Example CSV Output:
chemicalName,relationship,geneName,geneSymbol
"Genistein","Decreases expression","-","DEF2"
"Genistein","Increases expression","-","ABC1"
"Genistein","Decreases expression","tyrosine 3-monooxygenase","-"
"""
    return systemPrompt, mainPrompt


In [ ]:

def supplement_XLS_prompt(tableCSVstring, caption, sheet_name):
    """
    Returns systemPrompt, mainPrompt.
    """
    systemPrompt = (
        "You are an AI assistant specialized in extracting relationships between chemicals "
        "and genes from tables in scientific papers. Be precise, helpful, and honest. "
        "If a relationship is unclear or not present, state that clearly. Only use the data "
        "from the table and caption; do not assume anything.\n"
        "Do not extract relationships from negative or non-significant results."
    )

    mainPrompt = f"""### Extract Specific Relationships Between Chemicals and Genes from the following table in csv format with a header.

#### Table Caption:
{caption}

#### Table Sheet Name:
{sheet_name if sheet_name else "Not provided"}

#### Full Table:
{tableCSVstring}

---
### Provided Relationship List
The following relationships are predefined and must be used to describe chemical-gene interactions:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---
### Instructions
For each extracted relationship, generate a CSV object with the following fields:
- `chemicalName`: Name of the chemical.
- `relationship`: This must be chosen from the provided list of relationships. The relationship may be prefixed by `increase`, `decrease`, or `affect`.
- `geneName`: Gene name if provided, otherwise leave blank.
- `geneSymbol`: Gene symbol if provided, otherwise leave blank.

#### Guidelines:
1. Extract specific relationships between these chemicals and genes using only the predefined terms from the provided relationship list. Do not use any terms or descriptions of interactions from the source text unless they exactly match terms from the list.
2. Provide only the CSV output—no additional explanations or comments.
3. Ensure that your CSV output is well-formed and follows CSV format.
4. Recheck to ensure the relationship is in the provided relationship list.
5. Determining increase vs. decrease in expression—use ALL available context:
   - If sheet name is provided: Look for keywords like "up", "down", "increase", "decrease", "upregulated", "downregulated" in the sheet name.
   - If sheet name is not provided or unclear: Use the table caption and column headers:
     * Look for phrases like "up-regulated in [chemical] treatment" → chemical INCREASES expression
     * Look for phrases like "down-regulated in [chemical] treatment" → chemical DECREASES expression
     * Look for comparison direction in column headers
   - Fold change interpretation (when present):
     * Positive fold change typically means first condition > second condition
     * Negative fold change (or negative log2) typically means first condition < second condition
     * Always check column headers to understand what is being compared (e.g., "Control vs Treatment" or "Treatment vs Control")
   - If directionality is unclear: Use "affect expression" instead of "increase expression" or "decrease expression"
6. Do not extract relationships from negative or non-significant results. Only extract rows where a clear, significant effect is present.

---

### Example 1: With Sheet Name

#### Table Caption:
Genes with significant changes after bisphenol A treatment

#### Table Sheet Name:
The control decreased by more than 1.

#### Example Table:
ID,Gene Symbol,Description,log2 fold change
1,BRCA1,BReast CAncer gene 1,-2.7
2,TP53,tumor protein p53,-2.1

#### Example CSV Output:
chemicalName,relationship,geneName,geneSymbol
"bisphenol A","increase expression","BReast CAncer gene 1","BRCA1"
"bisphenol A","increase expression","tumor protein p53","TP53"

#### Logic: Sheet name indicates genes DOWN in control, so bisphenol A INCREASES expression.

---

### Example 2: Without Sheet Name (using caption)

#### Table Caption:
Genes up-regulated at least 1.5-fold in bisphenol A treatment in rats

#### Table Sheet Name:
Not provided

#### Example Table:
ID,Description,intensity, treatment
1,BReast CAncer gene 1,350,5000
2,tumor protein p53,200,500

#### Example CSV Output:
chemicalName,relationship,geneName,geneSymbol
"bisphenol A","increase expression","BReast CAncer gene 1",""
"bisphenol A","increase expression","tumor protein p53",""

#### Logic: Caption says "up-regulated in bisphenol A treatment" → bisphenol A INCREASES expression.

---

### Example 3: Ambiguous Direction

#### Table Caption:
Genes with altered expression after treatment

#### Table Sheet Name:
Not provided

#### Example Table:
Gene,Change
BRCA1,altered
TP53,modified

#### Example CSV Output:
chemicalName,relationship,geneName,geneSymbol
"treatment compound","affect expression","BRCA1",""
"treatment compound","affect expression","TP53",""

#### Logic: No clear directional information → use "affect expression".
"""

    return systemPrompt, mainPrompt


In [ ]:
def createTextPrompt(text, chem_map, gene_map):

    systemPrompt = """You are an AI assistant specialized in extracting relationships between chemicals and genes from scientific text. Be precise, accurate, and honest. Only extract relationships explicitly stated or clearly implied in the source text; if evidence is insufficient, do not extract the relationship.

Do not extract relationships from negative, null, or non-significant findings."""

    if isinstance(chem_map, str):
        chem_map = json.loads(chem_map)
    if isinstance(gene_map, str):
        gene_map = json.loads(gene_map)
    
    # Create a CSV-style list of chemicals: Name, MESH_ID
    chemicals = [f"{chem},{mesh_id}" for chem, mesh_id in chem_map.items()]
    
    # Create a CSV-style list of genes: Name, NCBI_ID
    genes = [f"{gene},{ncbi_id}" for gene, ncbi_id in gene_map.items()]
    
    # Join them into a string for the prompt
    chemicals_str = "\n".join(chemicals)
    genes_str = "\n".join(genes)
    
    mainPrompt = f"""### Task: Extract Chemical-Gene Relationships

Extract specific relationships between chemicals and genes from the given text. Format the output as CSV with one relationship per row.

#### Input Text:
{text}

#### Chemicals (Name, MESH ID):
{chemicals_str}

#### Genes (Name, NCBI ID):
{genes_str}

---
### Predefined Relationship Types

You must use ONLY the following relationship types. Do not use any other terms:

**Base relationships:**
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

**Directional prefixes:**
These relationships can be prefixed with one of the following to indicate direction:
- `increase`—when the text indicates upregulation, enhancement, stimulation, or increase
- `decrease`—when the text indicates downregulation, inhibition, reduction, or decrease
- `affect`—when the text indicates a change or effect but the direction is unclear or mixed

**Examples of valid relationships:**
- "increase expression"
- "decrease activity"
- "affect phosphorylation"
- "binding" (no prefix needed for non-directional relationships)

---
### Output Format

Generate a CSV with the following columns (include the header row):

**explanation, chemicalName, chemicalID, relationship, geneName, geneID, source**

#### Field Descriptions:
1. **explanation**: Brief description of the interaction (1–2 sentences max)
2. **chemicalName**: Exact name of the chemical from the provided list
3. **chemicalID**: MESH ID of the chemical
4. **relationship**: Relationship type from the predefined list (with prefix if applicable)
5. **geneName**: Exact name of the gene from the provided list
6. **geneID**: NCBI ID of the gene
7. **source**: The exact verbatim text snippet from the input that supports this relationship (use quotation marks if the snippet contains commas)

---
### Extraction Guidelines

DO:
1. Extract only relationships explicitly stated or clearly implied in the text
2. Use only the predefined relationship types from the list above
3. Map common verbs to the appropriate relationship:
   - "upregulates", "enhances", "stimulates" → increase
   - "downregulates", "inhibits", "suppresses", "reduces" → decrease
   - "modulates", "alters", "changes", "affects" → affect
4. Extract each distinct chemical-gene pair as a separate row
5. Use the exact chemical and gene names from the provided lists
6. Ensure all CSV fields are properly formatted and escaped

DO NOT:
1. Make assumptions about relationships not stated in the text
2. Use relationship terms not in the predefined list
3. Add interpretations or additional context beyond what is in the text
4. Extract vague or uncertain relationships
5. Include explanatory text outside the CSV format
6. Invent or paraphrase the source text—it must be verbatim
7. Extract negative relationships (e.g., "chemical X does NOT affect gene Y", "no effect on", "did not change")
8. Extract null findings or lack of effect—only extract positive findings where an interaction exists

---
### Example

#### Example Input Text:
"Exposure to cadmium chloride increases expression of MMP13 mRNA and MMP9 mRNA in cartilage cells. Bisphenol A reduces the activity of TOP2A protein and upregulates TP53 expression in chondrocytes. Treatment with lead acetate did not affect COL2A1 expression."

#### Example Output:
explanation,chemicalName,chemicalID,relationship,geneName,geneID,source
"cadmium chloride increases expression of MMP13","cadmium chloride","D019256","increase expression","MMP13","4332","Exposure to cadmium chloride increases expression of MMP13 mRNA and MMP9 mRNA in cartilage cells."
"cadmium chloride increases expression of MMP9","cadmium chloride","D019256","increase expression","MMP9","4318","Exposure to cadmium chloride increases expression of MMP13 mRNA and MMP9 mRNA in cartilage cells."
"bisphenol A decreases activity of TOP2A","bisphenol A","D001778","decrease activity","TOP2A","7153","Bisphenol A reduces the activity of TOP2A protein and upregulates TP53 expression in chondrocytes."
"bisphenol A increases expression of TP53","bisphenol A","D001778","increase expression","TP53","7157","Bisphenol A reduces the activity of TOP2A protein and upregulates TP53 expression in chondrocytes."

Note: The relationship "lead acetate did not affect COL2A1 expression" is NOT extracted because it describes a null finding (no effect).

---
### Your Task

Now extract all chemical-gene relationships from the input text provided above following these guidelines. Output only the CSV format with no additional commentary.
"""

    return systemPrompt, mainPrompt


In [ ]:
def createTableXML_prompt_updated(caption, tableText, tableXML, chem_map, gene_map):
    """
    Generates a system prompt and main prompt for a table (possibly from XML),
    including chemicals and genes. Returns systemPrompt, mainPrompt.
    """
    if isinstance(chem_map, str):
        chem_map = json.loads(chem_map)
    if isinstance(gene_map, str):
        gene_map = json.loads(gene_map)

    systemPrompt = (
        "You are an AI assistant specialized in extracting relationships between chemicals "
        "and genes from XML tables in scientific papers. Be precise, helpful, and honest. "
        "If a relationship is unclear or not present, state that clearly. Only use the data "
        "from the provided table; do not assume anything.\n"
        "Do not extract relationships from negative or non-significant results."
    )

    chemicals = [f"{chem},{mesh_id}" for chem, mesh_id in chem_map.items()]
    genes = [f"{gene},{ncbi_id}" for gene, ncbi_id in gene_map.items()]
    chemicals_str = "\n".join(chemicals)
    genes_str = "\n".join(genes)

    mainPrompt = f"""### Extract Specific Relationships Between Chemicals and Genes from the following table in XML format

#### Table Caption:
{caption}

#### Full Table:
{tableXML}

#### Chemicals and their MESH IDs:
{chemicals_str}

#### Genes and their NCBI IDs:
{genes_str}

---
### Provided Relationship List
The following relationships are predefined and must be used to describe chemical-gene interactions:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---
### Instructions
For each extracted relationship, generate a CSV object with the following fields:
- `chemicalName`: Name of the chemical
- `chemicalID`: Chemical ID if provided, otherwise leave blank.
- `relationship`: Must be chosen from the provided relationship list (may be prefixed by increase, decrease, or affect)
- `geneName`: Name of the gene
- `geneID`: Gene ID if provided, otherwise leave blank.

#### Guidelines:
1. Extract specific relationships included in the table.
2. Include ALL chemicals and genes in your output regardless of them being in the provided gene or chemical data. Set the ID field to - if not provided.
3. Provide only the CSV output—no additional explanations or comments.
4. Ensure that your CSV output is well-formed.
5. Recheck that the relationship is in the provided list.
6. In tabular data, relationships of interest are typically increase expression or decrease expression unless otherwise specified.
7. Do not infer relationships not supported by the provided content

---
### Example Output
chemicalName,chemicalID,relationship,geneName,geneID
"bisphenol A","C006780","increase expression","carbonyl reductase 1","29224"
"bisphenol A","C006780","increase expression","glucose-6-phosphate dehydrogenase","855480"
"""
    return systemPrompt, mainPrompt


In [ ]:
def validation_prompt_text(input_text, extracted_relationships):
    
    prompt = f"""Validate and correct relationships between chemicals and genes using ONLY the information provided below.

---

### Validation Rules

1. **Relationship Type:** Must match the predefined list. Replace invalid relationships with the closest valid alternative supported by the text; mark for removal (agreement=0) if no valid alternative exists. Chemical-only relationships (abundance, response to chemical, chemical synthesis) are invalid for chemical–gene pairs.

2. **Prefix Rules:** All relationships require a directional prefix (increase, decrease, affect) except cotreatment and binding. Use affect only when direction cannot be determined from the text. Remove any prefix applied to exempt relationships.

3. **Entity Validation:** Each relationship must involve exactly two entities. Set agreement=0 if entity information is missing or incomplete; do not guess missing entities.

4. **Null Relationships:** Remove interactions where the source text indicates no effect (e.g., "did not change", "no effect on", "failed to alter").

5. **Prior Work:** Remove interactions cited from previous studies or other works (e.g., "previous studies showed", "it has been reported", "as demonstrated by").

---

### Predefined Relationships:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Input:
- Original Text: {input_text}
- Extracted Relationships:
{extracted_relationships}

---

### Output Format
Return CSV with header: explanation, chemicalName, chemicalID, relationship, geneName, geneID, source, agreement, reasoning
- agreement: 1 (correct as-is) or 0 (needs correction/removal)
- reasoning: blank if agreement=1, otherwise specify which rule was violated and what changed

---

### Example
Original Text: "Exposure to cadmium chloride increases levels of MMP13 and MMP9. Bisphenol A reduces activity of TOP2A and affects expression of TP53 in chondrocytes. Lead acetate did not change COL2A1 expression. Previous studies showed that arsenic increases oxidation of KEAP1."

Input:
explanation,chemicalName,chemicalID,relationship,geneName,geneID,source
"cadmium chloride increases levels of MMP13","cadmium chloride","D019256","increase levels","MMP13","4332","Exposure to cadmium chloride increases levels of MMP13 and MMP9."
"cadmium chloride increases expression of MMP9","cadmium chloride","D019256","response to chemical","MMP9","4318","Exposure to cadmium chloride increases levels of MMP13 and MMP9."
"bisphenol A decreases activity of TOP2A","bisphenol A","D001778","decrease activity","TOP2A","7153","Bisphenol A reduces activity of TOP2A and affects expression of TP53 in chondrocytes."
"bisphenol A affects expression of TP53","bisphenol A","D001778","affect expression","TP53","7157","Bisphenol A reduces activity of TOP2A and affects expression of TP53 in chondrocytes."
"lead acetate has no effect on COL2A1","lead acetate","D007854","affect expression","COL2A1","1280","Lead acetate did not change COL2A1 expression."
"arsenic increases oxidation of KEAP1","arsenic","D001151","increase oxidation","KEAP1","9817","Previous studies showed that arsenic increases oxidation of KEAP1."

Output:
explanation,chemicalName,chemicalID,relationship,geneName,geneID,source,agreement,reasoning
"cadmium chloride increases expression of MMP13","cadmium chloride","D019256","increase expression","MMP13","4332","Exposure to cadmium chloride increases levels of MMP13 and MMP9.","0","Changed 'increase levels' to 'increase expression': 'levels' is not in the predefined list"
"cadmium chloride increases expression of MMP9","cadmium chloride","D019256","increase expression","MMP9","4318","Exposure to cadmium chloride increases levels of MMP13 and MMP9.","0","Changed 'response to chemical' to 'increase expression': chemical-only relationship invalid for chemical-gene pair"
"bisphenol A decreases activity of TOP2A","bisphenol A","D001778","decrease activity","TOP2A","7153","Bisphenol A reduces activity of TOP2A and affects expression of TP53 in chondrocytes.","1",""
"bisphenol A affects expression of TP53","bisphenol A","D001778","affect expression","TP53","7157","Bisphenol A reduces activity of TOP2A and affects expression of TP53 in chondrocytes.","1",""
"lead acetate has no effect on COL2A1","lead acetate","D007854","affect expression","COL2A1","1280","Lead acetate did not change COL2A1 expression.","0","Null relationship: source text states 'did not change'"
"arsenic increases oxidation of KEAP1","arsenic","D001151","increase oxidation","KEAP1","9817","Previous studies showed that arsenic increases oxidation of KEAP1.","0","Relationship from previous studies/other works"

---

Validate each row. Output only the CSV, no additional commentary.
"""
    return prompt


In [ ]:
def validation_prompt_table(caption, tableXML, extracted_relationships):
    
    prompt = f"""Validate and correct relationships between chemicals and genes extracted from a table.

---

### Validation Rules

1. **Relationship Type:** Must match the predefined list. Replace invalid relationships with the closest valid alternative supported by the table; mark for removal (agreement=0) if no valid alternative exists. Chemical-only relationships (abundance, response to chemical, chemical synthesis) are invalid for chemical–gene pairs.

2. **Prefix Rules:** All relationships require a directional prefix (increase, decrease, affect) except cotreatment and binding. Use table context to determine direction: positive values, or fold-change >1.0 indicate increase; negative values, or fold-change <1.0 indicate decrease.

3. **Entity Validation:** Each relationship must involve exactly two entities. Set agreement=0 if entity information is missing or incomplete.

4. **Null Relationships:** Remove interactions marked "NS", "no change", "0.00", empty effect cells, or fold-change = 1.0 with NS notation.

5. **Table Context:** Verify that the relationship type and prefix direction align with column headers, table symbols, and fold-change values.

---

### Predefined Relationships:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Input:
- Table Caption: {caption}
- Table XML:
{tableXML}
- Extracted Relationships:
{extracted_relationships}

---

### Output Format
Return CSV with header: chemicalName, chemicalID, relationship, geneName, geneID, agreement, reasoning
- agreement: 1 (correct as-is) or 0 (needs correction/removal)
- reasoning: blank if agreement=1, otherwise explain what changed and why

---

### Example

Input:
chemicalName,chemicalID,relationship,geneName,geneID
"cadmium chloride","D019256","increase levels","MMP13","4332"
"Lead acetate","D007854","affect expression","COL2A1","1280"

Table shows: Cadmium increases expression 2.3-fold; Lead shows 1.0 (NS).

Output:
chemicalName,chemicalID,relationship,geneName,geneID,agreement,reasoning
"cadmium chloride","D019256","increase expression","MMP13","4332","0","Changed 'increase levels' to 'increase expression': 'levels' not in predefined list"
"Lead acetate","D007854","affect expression","COL2A1","1280","0","Null relationship: table shows 1.0 (NS), no significant change"

---

Validate each row. Output only the CSV, no additional commentary.
"""
    return prompt


In [ ]:
def validation_prompt_figure(caption, extracted_relationships):
    
    prompt = f"""Validate and correct relationships between chemicals and genes extracted from a figure/image.

---

### Validation Rules

1. **Relationship Type:** Must match the predefined list. Replace invalid relationships with the closest valid alternative supported by the image; mark for removal (agreement=0) if no valid alternative exists. Chemical-only relationships (abundance, response to chemical, chemical synthesis) are invalid for chemical–gene pairs.

2. **Prefix Rules:** All relationships require a directional prefix (increase, decrease, affect) except cotreatment and binding. Ensure consistent lowercase capitalization.

3. **Image Direction Indicators:** Use visual cues to determine directionality: taller/darker bars, upward slopes, symbols, and warm colors indicate increase; shorter/lighter bars, downward slopes, symbols, and cool colors indicate decrease.

4. **Entity Validation:** Each relationship must involve exactly two entities. Use "-" for missing gene name or symbol. Set agreement=0 if entity information is incomplete.

5. **Null Relationships:** Remove interactions marked "NS" or "n.s.", showing no visible difference, or indicating no effect.

---

### Predefined Relationships:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Input:
- Image Caption: {caption}
- Extracted Relationships:
{extracted_relationships}

---

### Output Format
Return CSV with header: chemicalName, relationship, geneName, geneSymbol, agreement, reasoning
- agreement: 1 (correct as-is) or 0 (needs correction/removal)
- reasoning: blank if agreement=1, otherwise explain what changed and why

---

### Example

Input:
chemicalName,relationship,geneName,geneSymbol
"Genistein","Decreases expression","-","DEF2"
"Genistein","Increases expression","-","ABC1"
"Genistein","activity","tyrosine 3-monooxygenase","-"

Caption: Gene regulation by Genistein.

Output:
chemicalName,relationship,geneName,geneSymbol,agreement,reasoning
"Genistein","decrease expression","-","DEF2","0","Fixed capitalization: 'Decreases' to 'decrease'"
"Genistein","increase expression","-","ABC1","0","Fixed capitalization: 'Increases' to 'increase'"
"Genistein","affect activity","tyrosine 3-monooxygenase","-","0","Added 'affect' prefix: 'activity' requires a directional prefix"

---

Validate each row. Output only the CSV, no additional commentary.
"""
    return prompt


In [ ]:
def validation_prompt_figure_table(caption, extracted_relationships):
    
    prompt = f"""Validate and correct relationships between chemicals and genes extracted from a table in a figure/image.

---

### Validation Rules

1. **Relationship Type:** Must match the predefined list. Replace invalid relationships with the closest valid alternative supported by the table; mark for removal (agreement=0) if no valid alternative exists. Chemical-only relationships (abundance, response to chemical, chemical synthesis) are invalid for chemical–gene pairs.

2. **Prefix Rules:** All relationships require a directional prefix (increase, decrease, affect) except cotreatment and binding. Ensure consistent lowercase capitalization.

3. **Direction Validation:** Verify prefix direction using all available context:
   - Caption: "up-regulated" → increase; "down-regulated" → decrease.
   - Fold-change: >1.0 or positive log2 = increase; <1.0 or negative log2 = decrease (verify comparison direction from column headers).

4. **Entity Validation:** Each relationship must involve exactly two entities. Use "-" for missing gene name or symbol. Set agreement=0 if entity information is incomplete.

5. **Null Relationships:** Remove interactions marked "NS", "n.s.", fold-change = 1.0, or indicating no effect.

---

### Predefined Relationships:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Input:
- Table Caption: {caption}
- Extracted Relationships:
{extracted_relationships}

---

### Output Format
Return CSV with header: chemicalName, relationship, geneName, geneSymbol, agreement, reasoning
- agreement: 1 (correct as-is) or 0 (needs correction/removal)
- reasoning: blank if agreement=1, otherwise explain what changed and why

---

### Example

Input:
chemicalName,relationship,geneName,geneSymbol
"Genistein","Decreases expression","-","DEF2"
"Genistein","Increases expression","-","ABC1"
"Genistein","Decreases expression","tyrosine 3-monooxygenase","-"

Caption: Gene regulation by Genistein.

Output:
chemicalName,relationship,geneName,geneSymbol,agreement,reasoning
"Genistein","decrease expression","-","DEF2","0","Fixed capitalization: 'Decreases' to 'decrease'"
"Genistein","increase expression","-","ABC1","0","Fixed capitalization: 'Increases' to 'increase'"
"Genistein","decrease expression","tyrosine 3-monooxygenase","-","0","Fixed capitalization: 'Decreases' to 'decrease'"

---

Validate each row. Output only the CSV, no additional commentary.
"""
    return prompt


In [ ]:
def validation_prompt_supplement(caption, sheet_name, tableCSVstring, extracted_relationships):
    
    prompt = f"""Validate and correct relationships between chemicals and genes extracted from a supplemental table.

---

### Validation Rules

1. **Relationship Type:** Must match the predefined list. Replace invalid relationships with the closest valid alternative supported by the table; mark for removal (agreement=0) if no valid alternative exists. Chemical-only relationships (abundance, response to chemical, chemical synthesis) are invalid for chemical–gene pairs.

2. **Prefix Rules:** All relationships require a directional prefix (increase, decrease, affect) except cotreatment and binding. Use affect when direction is unclear.

3. **Direction Validation:** Verify prefix direction using all available context:
   - Sheet name: "down" or "decrease" in sheet name → genes DOWN in control → chemical INCREASES expression; "up" or "increase" → chemical DECREASES expression.
   - Caption/headers: "up-regulated in [chemical]" → increase; "down-regulated in [chemical]" → decrease.
   - Fold-change: >1.0 or positive log2 = increase; <1.0 or negative log2 = decrease (verify comparison direction from column headers).

4. **Entity Validation:** Each relationship must involve exactly two entities. Set agreement=0 if entity information is missing or incomplete.

5. **Null Relationships:** Remove interactions marked "NS", "no change", fold-change = 1.0, or ~0 in log scale.

---

### Predefined Relationships:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Input:
- Table Caption: {caption}
- Sheet Name: {sheet_name if sheet_name else "Not provided"}
- Table CSV:
{tableCSVstring}
- Extracted Relationships:
{extracted_relationships}

---

### Output Format
Return CSV with header: chemicalName, relationship, geneName, geneSymbol, agreement, reasoning
- agreement: 1 (correct as-is) or 0 (needs correction/removal)
- reasoning: blank if agreement=1, otherwise explain what changed and why

---

### Example

Caption: Genes with significant changes after bisphenol A treatment.
Sheet Name: Control decreased.

Input:
chemicalName,relationship,geneName,geneSymbol
"bisphenol A","decrease expression","carbonyl reductase 1","CR1"
"bisphenol A","increase expression","glucose-6-phosphate dehydrogenase","G6PD"

Output:
chemicalName,relationship,geneName,geneSymbol,agreement,reasoning
"bisphenol A","increase expression","carbonyl reductase 1","CR1","0","Changed 'decrease' to 'increase' sheet name indicates genes down in control"
"bisphenol A","increase expression","glucose-6-phosphate dehydrogenase","G6PD","1",""

---

Validate each row. Output only the CSV, no additional commentary.
"""
    return prompt


In [ ]:
systemPromptValidateText = """
You are an AI assistant responsible for validating extracted relationships between chemicals and genes based solely on a provided source text. Verify whether each extracted relationship is factually supported by the text and conforms to the allowed relationship schema.

Constraints:
- Do NOT add new relationships or remove existing relationships.
- Correct a relationship ONLY if it is factually incorrect, unsupported by the text, or violates the relationship rules.
- Do NOT introduce external knowledge or assumptions.
- You may ONLY modify: relationship, explanation.
"""

systemPromptValidateTable = """
You are an AI assistant responsible for validating extracted relationships between chemicals and genes based solely on a provided XML table. Verify whether each extracted relationship is factually supported by the table and conforms to the allowed relationship schema.

Constraints:
- Do NOT add new relationships or remove existing relationships.
- Correct a relationship ONLY if it is factually incorrect, unsupported by the table, or violates the relationship rules.
- Do NOT introduce external knowledge or assumptions.
- You may ONLY modify: chemicalName, chemicalID, relationship, geneName, geneSymbol.
"""

systemPromptValidateFigure = """
You are an AI assistant responsible for validating extracted relationships between chemicals and genes based solely on a provided figure/image and its caption. Verify whether each extracted relationship is factually supported by the visual data and caption, and conforms to the allowed relationship schema.

Constraints:
- Do NOT add new relationships or remove existing relationships.
- Correct a relationship ONLY if it is factually incorrect, unsupported by the image/caption, or violates the relationship rules.
- Do NOT introduce external knowledge or assumptions.
- You may ONLY modify: chemicalName, relationship, geneName, geneSymbol.
"""

systemPromptValidateFigureTable = """
You are an AI assistant responsible for validating extracted relationships between chemicals and genes based solely on a provided table embedded in a figure/image and its caption. Verify whether each extracted relationship is factually supported by the table data and caption, and conforms to the allowed relationship schema.

Constraints:
- Do NOT add new relationships or remove existing relationships.
- Correct a relationship ONLY if it is factually incorrect, unsupported by the table/caption, or violates the relationship rules.
- Do NOT introduce external knowledge or assumptions.
- You may ONLY modify: chemicalName, relationship, geneName, geneSymbol.
"""

systemPromptValidateSupplement = """
You are an AI assistant responsible for validating extracted relationships between chemicals and genes based solely on a provided supplemental table (CSV format), its caption, and sheet name. Verify whether each extracted relationship is factually supported by the table data and conforms to the allowed relationship schema.

Constraints:
- Do NOT add new relationships or remove existing relationships.
- Correct a relationship ONLY if it is factually incorrect, unsupported by the table data, or violates the relationship rules.
- Do NOT introduce external knowledge or assumptions.
- Pay special attention to sheet names and captions for directional information (increase/decrease).
- You may ONLY modify: chemicalName, relationship, geneName, geneSymbol.
"""


In [ ]:

import pandas as pd
import json
from pathlib import Path

# Input and output
input_csv = Path(output_data_path) / "validation_combined.csv"
output_csv = Path(output_data_path) / "validation_combined_prompts_updated.csv"

df = pd.read_csv(input_csv, dtype=str, keep_default_na=False)

# Only these columns will be modified
prompt_cols = [
    "main_prompt",
    "main_prompt_val_gpt",
    "main_prompt_val_deepseek",
    "main_prompt_val_claude",
    "main_prompt_val_gpt4o",
]

# Each validation prompt uses the corresponding extractor's output
validation_mapping = {
    "main_prompt_val_gpt": "gpt_E_output",
    "main_prompt_val_deepseek": "deepseek_E_output",
    "main_prompt_val_claude": "claude_E_output",
    "main_prompt_val_gpt4o": "gpt4o_E_output",
}

# Keep a copy for verification
original = df.drop(columns=prompt_cols).copy()

counts = {"TABLE": 0, "SUPPLEMENT": 0}

for idx, row in df.iterrows():
    section = row["section_type"].strip().upper()

    if section not in counts:
        continue

    caption = row["caption"]
    counts[section] += 1

    if section == "TABLE":
        # Regenerate table extraction prompt
        chem_map = json.loads(row["chemical_map"]) if row["chemical_map"] else {}
        gene_map = json.loads(row["gene_map"]) if row["gene_map"] else {}

        _, new_main_prompt = createTableXML_prompt_updated(
            caption,
            row["text"],
            row["table_xml"],
            chem_map,
            gene_map
        )

    else:  # SUPPLEMENT
        # Recover the original CSV table content BEFORE replacing the prompt
        tableCSVstring = extract_full_table_csv(row["main_prompt"])

        if not tableCSVstring:
            raise ValueError(
                f"Missing supplement table at row {idx}, PMID {row['pmid']}"
            )

        _, new_main_prompt = supplement_XLS_prompt(
            tableCSVstring,
            caption,
            row["sheet_name"]
        )

    df.at[idx, "main_prompt"] = new_main_prompt

    # Regenerate validation prompts using existing model outputs
    for prompt_col, extraction_col in validation_mapping.items():
        extracted = row[extraction_col]

        if not extracted.strip():
            df.at[idx, prompt_col] = ""
            continue

        if section == "TABLE":
            new_val_prompt = validation_prompt_table(
                caption,
                row["table_xml"],
                extracted
            )
        else:
            new_val_prompt = validation_prompt_supplement(
                caption,
                row["sheet_name"],
                tableCSVstring,
                extracted
            )

        df.at[idx, prompt_col] = new_val_prompt

# Verify no other fields were changed
assert df.drop(columns=prompt_cols).equals(original)
assert len(df) == len(original)

# Save without overwriting original
df.to_csv(output_csv, index=False)

print("Updated TABLE rows:", counts["TABLE"])
print("Updated SUPPLEMENT rows:", counts["SUPPLEMENT"])
print("Total rows preserved:", len(df))
print("All existing model outputs preserved.")
print("Saved to:", output_csv)


In [ ]:
"""
process_supplements.py
----------------------
Scans a single flat folder for all *_supplements_metadata.csv files,
then for each metadata file processes every referenced supplement
(XLS, XLSX, CSV), batches each sheet, and creates system+main prompts.

Compatible with Python 3.7+
"""


# ── CONFIG ────────────────────────────────────────────────────────────────────
FOLDER     = supplement_path   # <-- your path variable
BATCH_SIZE = 30
OUTPUT_CSV = None              # set to a filename string to save, e.g. "all_supp_prompts.csv"
# ──────────────────────────────────────────────────────────────────────────────

# supplement_XLS_prompt is defined elsewhere in the notebook and called directly


def find_file(folder, filename):
    # type: (str, str) -> Optional[str]
    """
    Find a supplement file in the folder.
    Tries original filename first, then dots-replaced-by-underscores variant.
    (metadata may store 'pone.0000097.s039.xls'; disk may have 'pone_0000097_s039.xls')
    """
    base, ext  = os.path.splitext(filename)
    underscore = base.replace(".", "_") + ext
    for candidate in [filename, underscore]:
        path = os.path.join(folder, candidate)
        if os.path.exists(path):
            return path
    return None


def read_sheets(filepath):
    # type: (str) -> Dict[str, pd.DataFrame]
    """Read all sheets from an XLS / XLSX / CSV. Returns {sheet_name: df}."""
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(filepath)
        return {os.path.splitext(os.path.basename(filepath))[0]: df}
    elif ext == ".xls":
        xls = pd.ExcelFile(filepath, engine="xlrd")
        return {s: pd.read_excel(filepath, sheet_name=s, engine="xlrd")
                for s in xls.sheet_names}
    elif ext in (".xlsx", ".xlsm"):
        xls = pd.ExcelFile(filepath, engine="openpyxl")
        return {s: pd.read_excel(filepath, sheet_name=s, engine="openpyxl")
                for s in xls.sheet_names}
    else:
        print("    ⚠ Unsupported extension '{}' — skipping".format(ext))
        return {}


def create_batches(df, batch_size):
    # type: (pd.DataFrame, int) -> List[str]
    """Split a DataFrame into CSV-string chunks (header included, no index)."""
    return [
        df.iloc[i:i + batch_size].to_csv(index=False)
        for i in range(0, len(df), batch_size)
    ]


def create_supp_prompts(batches, caption, sheet_name, pmid, filename):
    # type: (List[str], str, str, int, str) -> pd.DataFrame
    """Create one prompt record per batch."""
    records = []
    for batch_csv in batches:
        system_prompt, main_prompt = supplement_XLS_prompt(batch_csv, caption, sheet_name)
        records.append({
            "pmid":          pmid,
            "caption":       caption,
            "sheet_name":    sheet_name,
            "section_type":  "SUPPLEMENT",
            "system_prompt": system_prompt,
            "main_prompt":   main_prompt,
            "source_file":   filename,
        })
    return pd.DataFrame(records)


def run_pipeline(folder, batch_size=BATCH_SIZE):
    # type: (str, int) -> pd.DataFrame

    pattern        = os.path.join(folder, "*_supplements_metadata.csv")
    metadata_files = sorted(glob.glob(pattern))

    if not metadata_files:
        print("No *_supplements_metadata.csv files found in: {}".format(folder))
        return pd.DataFrame()

    print("Found {} metadata file(s)\n".format(len(metadata_files)))

    all_prompts   = []
    missing_files = []

    for meta_path in metadata_files:
        meta_name = os.path.basename(meta_path)
        print("── {}".format(meta_name))

        meta = pd.read_csv(meta_path)

        for _, row in meta.iterrows():
            pmid     = row["pmid"]
            filename = row["filename"]
            caption  = row["caption"]

            filepath = find_file(folder, filename)
            if filepath is None:
                print("   ⚠ Not found: {}".format(filename))
                missing_files.append(filename)
                continue

            print("   ✓ {}  (pmid={})".format(filename, pmid))
            sheets = read_sheets(filepath)

            for sheet_name, df in sheets.items():
                n_batches = -(-df.shape[0] // batch_size)  # ceiling division
                print("      → sheet '{}'  {} rows → {} batch(es)".format(
                    sheet_name, df.shape[0], n_batches))
                batches    = create_batches(df, batch_size)
                prompts_df = create_supp_prompts(
                    batches, caption, sheet_name, pmid, filename)
                all_prompts.append(prompts_df)

        print()

    if not all_prompts:
        print("No data processed.")
        return pd.DataFrame()

    combined = pd.concat(all_prompts, ignore_index=True)

    print("{}".format("─" * 50))
    print("✅ {} prompt rows  |  {} sheet(s) processed  |  {} file(s) missing".format(
        len(combined), len(all_prompts), len(missing_files)))
    if missing_files:
        print("Missing files:")
        for f in missing_files:
            print("  - {}".format(f))

    return combined


# ── RUN ───────────────────────────────────────────────────────────────────────
combined_df = run_pipeline(FOLDER, BATCH_SIZE)
combined_df.head()


### Build prompts for figures, text, tables, and supplements


In [ ]:
# create fig prompts
def parse_map(cell):
    """
    Converts a newline-separated 'name: ID' string into a dict.
    Returns None if cell is NaN or empty.
    """
    if pd.isna(cell) or not str(cell).strip():
        return None

    result = {}
    for line in str(cell).split("\n"):
        if ":" in line:
            name, _id = line.split(":", 1)
            result[name.strip()] = _id.strip()
    return result if result else None

def build_prompts(row):
    chem_map = parse_map(row["chemical_map"])
    gene_map = parse_map(row["gene_map"])
    caption = row["caption"]

    if row["type"] == "figure":
        sys_prompt, main_prompt = fig_prompt(
            # chem_map=chem_map,
            # gene_map=gene_map,
            caption=caption
        )
    if row["type"] == "table":
        sys_prompt, main_prompt = fig_table_prompt(
            # chem_map=chem_map,
            # gene_map=gene_map,
            caption=caption
        )
 
    df = pd.Series({
        "system_prompt": sys_prompt,
        "main_prompt": main_prompt
    })

    return df

figs[["system_prompt", "main_prompt"]] = figs.apply(build_prompts, axis=1)
figs['section_type'] = 'FIG'
len(figs)


In [ ]:
# create text & table prompts
def generate_prompts(row):
    # Skip TABLE or FIG rows
    chem_map = row['chem_map']
    gene_map = row['gene_map']
    
    if row['section_type'] == 'TABLE':
        return createTableXML_prompt_updated(row['table_caption'], row['table_text'], row['table_xml'], chem_map, gene_map)

    # Convert chem_map and gene_map from JSON strings to dicts if needed
    if isinstance(chem_map, str):
        chem_map = json.loads(chem_map)
    if isinstance(gene_map, str):
        gene_map = json.loads(gene_map)

    # Call your existing function that returns a tuple (system_prompt, main_prompt)
    return createTextPrompt(row['text'], chem_map, gene_map)

# Apply to the DataFrame
prompts = text.apply(generate_prompts, axis=1)

# Split the tuple into two separate columns
text['system_prompt'] = prompts.apply(lambda x: x[0] if x is not None else None)
text['main_prompt'] = prompts.apply(lambda x: x[1] if x is not None else None)

len(text)


In [ ]:
# combine

text = text.rename(columns={"chem_map": "chemical_map"})
text = text.rename(columns={"table_caption": "caption"})

combined_main = pd.concat(
    [figs, text, combined_df],
    ignore_index=True
)
combined = combined_main.drop(columns = ['id', 'pmcid', 'table_text', 'number'], axis=1)


In [ ]:
combined = combined.drop(
    combined[
        (combined["section_type"] == "TABLE") &
        (combined["table_xml"].isna() | (combined["table_xml"].str.strip() == ""))
    ].index
)
combined
combined.to_csv(f'{input_data_path}/input_combined.csv')


## 6. Extraction (paid API calls, resumable)


In [ ]:

# API clients. Keys come from environment variables; nothing is hard-coded.
def _require_env(name):
    value = os.environ.get(name)
    if not value:
        raise EnvironmentError(f"Set the {name} environment variable before running.")
    return value

deepseek_key = ''
anthropic_key = ''
gpt_key = ''

bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)
openai_client = OpenAI(api_key=gpt_key)
claude_client = Anthropic(api_key=anthropic_key)
deepseek_client = OpenAI(
    api_key=deepseek_key,
    base_url="https://api.deepseek.com",
)

# Extraction models. Each name is used as a column prefix: <name>_output.
MODELS = ['claude', 'deepseek', 'gpt', 'gpt4o']


In [ ]:
input = pd.read_csv(f'{input_data_path}/input_combined.csv').drop(columns=['Unnamed: 0'], errors='ignore')

for name in MODELS:
    col = f"{name}_output"
    if col not in input.columns:
        input[col] = None

print(input.shape)


In [ ]:
def image_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


In [ ]:

# Model call wrappers. Each returns the model's text, or None / "ERROR: ..." on failure.

def call_claude(system_prompt, main_prompt, claude_model="claude-sonnet-4-5", image_path=None):
    """Claude Messages API with an optional figure image."""
    messages = []

    if image_path:
        messages.append({
            "role": "user",
            "content": [{
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/jpeg",  # figure files are jpg
                    "data": image_to_base64(image_path),
                },
            }],
        })

    messages.append({
        "role": "user",
        "content": [{"type": "text", "text": main_prompt}],
    })

    try:
        response = claude_client.messages.create(
            model=claude_model,
            system=system_prompt,
            messages=messages,
            max_tokens=10000,
            temperature=0,
        )

        text_blocks = [
            block.text
            for block in (getattr(response, "content", None) or [])
            if hasattr(block, "text") and block.text
        ]
        response_text = "\n".join(text_blocks)

        if not response_text.strip():
            response_text = "ERROR: Empty Claude response"

        return response_text

    except Exception as e:
        return f"ERROR: {e}"


def call_deepseek(system_prompt, main_prompt, model="deepseek-flash"):
    """DeepSeek chat completion. Returns the text, or None on failure."""
    try:
        response = deepseek_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": main_prompt},
            ],
            temperature=0.0,
            seed=42,
            max_tokens=8192,
            stream=False,
        )
        return response.choices[0].message.content

    except Exception as e:
        print(f"✗ DeepSeek call failed: {e}")
        return None


def call_llama(system_prompt, main_prompt, model="meta.llama3-2-90b-instruct-v1:0"):
    """Llama on Bedrock. Raises on failure (the caller catches it)."""
    body = {
        "prompt": f"<<SYS>>{system_prompt}<</SYS>>\n\n{main_prompt}",
        "max_gen_len": 10000,
        "temperature": 0,
    }

    response = bedrock.invoke_model(modelId=model, body=json.dumps(body))
    payload = json.loads(response["body"].read())
    return payload["generation"]


def _call_openai_responses(main_prompt, model, image_path=None):
    content = []

    if image_path:
        content.append({
            "type": "input_image",
            "image_url": f"data:image/png;base64,{image_to_base64(image_path)}",
        })

    content.append({"type": "input_text", "text": main_prompt})

    response = openai_client.responses.create(
        model=model,
        input=[{"role": "user", "content": content}],
        max_output_tokens=10000,
    )
    return response.output_text


# NOTE: call_gpt and call_gpt4o do not send system_prompt. The original notebook
# also did this. It is kept unchanged here because it affects results.
def call_gpt(system_prompt, main_prompt, image_path=None, model="gpt-4.1"):
    return _call_openai_responses(main_prompt, model, image_path)


def call_gpt4o(system_prompt, main_prompt, image_path=None, model="gpt-4o"):
    return _call_openai_responses(main_prompt, model, image_path)


In [ ]:
# ============================================================
# CHECK OUTPUT STATUS
# ============================================================

def get_run_status(value):
    if value is None:
        return "run"

    if isinstance(value, str):
        if not value.strip():
            return "run"
        if "ERROR" in value:
            return "re-run"
        return "skip"

    if pd.isna(value):
        return "run"

    return "skip"


# ============================================================
# RUN MODELS ON ROW
# ============================================================

def run_models_on_row(row, idx):
    is_fig = row.get("section_type") in ["FIG", "FIG_TABLE"]

    image_path = (
        f"{figures_path}/{row['filename']}"
        if is_fig and pd.notna(row.get("filename"))
        else None
    )

    print(f"\n🔹 Row {idx} | section_type={row['section_type']}")
    print(image_path)

    outputs = {}

    # ---------- Claude ----------
    status = get_run_status(row.get("claude_output"))

    if status == "skip":
        print("  ⏭ Claude skipped (output exists)")
        outputs["claude_output"] = row["claude_output"]
    else:
        print(f"  {'🔄' if status == 're-run' else '▶'} Claude {status}...")
        try:
            outputs["claude_output"] = call_claude(
                row["system_prompt"],
                row["main_prompt"],
                image_path=image_path
            )
            print("    ✓ Claude done")
        except Exception as e:
            outputs["claude_output"] = f"ERROR: {e}"
            print("    ✗ Claude failed")

    # ---------- GPT ----------
    status = get_run_status(row.get("gpt_output"))

    if status == "skip":
        print("  ⏭ GPT skipped (output exists)")
        outputs["gpt_output"] = row["gpt_output"]
    else:
        print(f"  {'🔄' if status == 're-run' else '▶'} GPT {status}...")
        try:
            outputs["gpt_output"] = call_gpt(
                row["system_prompt"],
                row["main_prompt"],
                image_path=image_path
            )
            print("    ✓ GPT done")
        except Exception as e:
            outputs["gpt_output"] = f"ERROR: {e}"
            print("    ✗ GPT failed")

    # ---------- GPT4o ----------
    status = get_run_status(row.get("gpt4o_output"))

    if status == "skip":
        print("  ⏭ GPT4o skipped (output exists)")
        outputs["gpt4o_output"] = row["gpt4o_output"]
    else:
        print(f"  {'🔄' if status == 're-run' else '▶'} GPT4o {status}...")
        try:
            outputs["gpt4o_output"] = call_gpt4o(
                row["system_prompt"],
                row["main_prompt"],
                image_path=image_path
            )
            print("    ✓ GPT4o done")
        except Exception as e:
            outputs["gpt4o_output"] = f"ERROR: {e}"
            print("    ✗ GPT4o failed")

    # ---------- DeepSeek ----------
    if is_fig:
        print("  ⏭ DeepSeek skipped (FIG)")
        outputs["deepseek_output"] = row.get("deepseek_output")
    else:
        status = get_run_status(row.get("deepseek_output"))

        if status == "skip":
            print("  ⏭ DeepSeek skipped (output exists)")
            outputs["deepseek_output"] = row.get("deepseek_output")
        else:
            print(f"  {'🔄' if status == 're-run' else '▶'} DeepSeek {status}...")
            try:
                outputs["deepseek_output"] = call_deepseek(
                    row["system_prompt"],
                    row["main_prompt"]
                )
                print("    ✓ DeepSeek done")
            except Exception as e:
                outputs["deepseek_output"] = f"ERROR: {e}"
                print("    ✗ DeepSeek failed")

    # ---------- LLaMA ----------
    if is_fig:
        print("  ⏭ LLaMA skipped (FIG)")
        outputs["llama_output"] = row.get("llama_output")
    else:
        status = get_run_status(row.get("llama_output"))

        if status == "skip":
            print("  ⏭ LLaMA skipped (output exists)")
            outputs["llama_output"] = row.get("llama_output")
        else:
            print(f"  {'🔄' if status == 're-run' else '▶'} LLaMA {status}...")
            try:
                outputs["llama_output"] = call_llama(
                    row["system_prompt"],
                    row["main_prompt"]
                )
                print("    ✓ LLaMA done")
            except Exception as e:
                outputs["llama_output"] = f"ERROR: {e}"
                print("    ✗ LLaMA failed")

    print(f"✅ Row {idx} complete")
    return pd.Series(outputs)


In [ ]:

# Main extraction loop. Results are written into `output`, which is the same
# DataFrame as `input`. (The original referenced `output` without defining it.)
output_file = f"{output_data_path}/extraction_combined.csv"
output = input

for idx, row in input.iterrows():
    is_fig = row.get("section_type") in ["FIG", "FIG_TABLE"]

    # FIG rows do not run DeepSeek or LLaMA, so they are left out of the skip check.
    cols_to_check = ["claude_output", "gpt_output", "gpt4o_output"]
    if not is_fig:
        cols_to_check += ["deepseek_output", "llama_output"]

    if all(get_run_status(row.get(col)) == "skip" for col in cols_to_check):
        print(f"Row {idx} skipped: outputs already exist")
        continue

    outputs = run_models_on_row(row, idx)
    for col, val in outputs.items():
        output.at[idx, col] = val

    if (idx + 1) % 50 == 0:
        output.to_csv(output_file, index=False)
        print(f"Checkpoint saved at row {idx + 1}")

output.to_csv(output_file, index=False)
print("\nAll rows complete and saved to", output_file)


## 7. Validation


In [ ]:
input = pd.read_csv(f'{output_data_path}/extraction_combined.csv')
input.columns


In [ ]:
def find_csv_blocks(text):
    pattern = r'(?:^[^\n,]+(?:,[^\n,]+)+\n)+(?:^[^\n,]+(?:,[^\n,]+)+$)'
    matches = re.findall(pattern, text, re.MULTILINE)
    return matches

def extract_csv(text):

    # ✅ Handle None / NaN / non-strings safely
    if text is None or pd.isna(text):
        return None

    if not isinstance(text, str):
        text = str(text)

    # STEP 1: Look inside ``` or ```csv blocks
    code_block_pattern = r"```(?:csv)?\s*\n(.*?)```"
    code_blocks = re.findall(code_block_pattern, text, re.DOTALL | re.IGNORECASE)

    csv_blocks = []

    for block in code_blocks:
        if "," in block and len(block.strip().splitlines()) >= 2:
            csv_blocks.append(block.strip())

    if csv_blocks:
        return "\n\n".join(csv_blocks)

    # STEP 2: fallback regex detection
    fallback = find_csv_blocks(text)
    if fallback:
        return "\n\n".join(fallback)

    return None

llm_cols = ['deepseek_output', 'gpt_output', 'claude_output', 'gpt4o_output']

for col in llm_cols:
    new_col = col.replace("_output", "_E_output")
    print(f"Processing {col} → {new_col}")

    input[new_col] = input[col].apply(extract_csv)

input


In [ ]:
# build validation prompts for each
# add system prompts based on section_type
def assign_validation_prompt(section_type):
    if not isinstance(section_type, str):
        return systemPromptValidateText  # safe default

    st = section_type.strip().upper()

    if "SUPPLEMENT" in st:
        return systemPromptValidateSupplement

    if "FIG" in st and "TABLE" in st:
        return systemPromptValidateFigureTable

    if "FIG" in st:
        return systemPromptValidateFigure

    if "TABLE" in st:
        return systemPromptValidateTable

    # default = regular text validation
    return systemPromptValidateText

input["system_prompt_val"] = input["section_type"].apply(assign_validation_prompt)


In [ ]:
def extract_full_table_csv(text: str) -> str:
    """
    Extracts the CSV content after '### Full Table:' until the next '---' line.
    Returns the CSV string or None if not found.
    """
    if not isinstance(text, str):
        return None

    lines = text.splitlines()
    table_lines = []
    in_table = False

    for line in lines:
        stripped = line.strip()
        # Start collecting after Full Table:
        if not in_table and stripped.lower().startswith("#### full table:"):
            in_table = True
            continue
        # Stop at section delimiter
        if in_table and stripped.startswith("---"):
            break
        # Collect lines if inside table
        if in_table:
            table_lines.append(line)

    if not table_lines:
        return None

    return "\n".join(table_lines).strip()


In [ ]:

def build_main_prompt_val(row, extracted_col):
    """Validation main prompt for one row, validating the extraction in extracted_col
    (for example 'gpt_E_output'). Returns None when that extraction is missing."""
    section_type = str(row.get("section_type", "")).strip().upper()

    if pd.isna(row.get(extracted_col)):
        return None

    extracted_relationships = row.get(extracted_col, "")
    caption = row.get("caption", "")
    text = row.get("text", "")
    table_xml = row.get("table_xml", "")
    sheet_name = row.get("sheet_name", "")

    tableCSVstring = ''
    if section_type == "SUPPLEMENT":
        tableCSVstring = extract_full_table_csv(row.get('main_prompt'))

    if "SUPPLEMENT" in section_type:
        return validation_prompt_supplement(caption, sheet_name, tableCSVstring, extracted_relationships)

    if "FIG" in section_type and "TABLE" in section_type:
        return validation_prompt_figure_table(caption, extracted_relationships)

    if "FIG" in section_type:
        return validation_prompt_figure(caption, extracted_relationships)

    if "TABLE" in section_type:
        return validation_prompt_table(caption, table_xml, extracted_relationships)

    # Default: text
    return validation_prompt_text(text, extracted_relationships)


In [ ]:

input["main_prompt_val_gpt"] = input.apply(lambda r: build_main_prompt_val(r, "gpt_E_output"), axis=1)
input["main_prompt_val_deepseek"] = input.apply(lambda r: build_main_prompt_val(r, "deepseek_E_output"), axis=1)
input["main_prompt_val_claude"] = input.apply(lambda r: build_main_prompt_val(r, "claude_E_output"), axis=1)
input["main_prompt_val_gpt4o"] = input.apply(lambda r: build_main_prompt_val(r, "gpt4o_E_output"), axis=1)


In [ ]:
# List of new columns to add
new_cols = [
    'deepseekV_gptE', 'deepseekV_claudeE',
    'gptV_deepseekE', 'gptV_claudeE',
    'claudeV_deepseekE', 'claudeV_gptE',
    'deepseekV_gpt4oE', 'claudeV_gpt4oE',
    'gpt4oV_deepseekE', 'gpt4oV_claudeE'
]

# Add each column with None as default
for col in new_cols:
    if col not in input.columns:
        input[col] = None

# input.to_csv(f"{input_data_path}/input_val_combined.csv", index=False)


In [ ]:
# NOTE: this reloads input_val_combined.csv. Prompt columns built in the cells above are
# replaced by whatever is saved on disk. The save in the cell above is commented out.
# Verify
input = pd.read_csv(f"{input_data_path}/input_val_combined.csv")
input.columns


### Validators
Four validators, one per extraction source. Each has its own skip rule (per column for DeepSeek and Claude, per row for GPT and GPT-4o), so they are kept separate.


In [ ]:
def run_validation_models_deepseek(row, idx):
    """
    Calls DeepSeek validation on extracted prompts for a single row.
    Outputs:
        - deepseekV_claudeE
        - deepseekV_gptE
        - deepseekV_gpt4oE
    Skips FIG/FIG_TABLE rows (DeepSeek can't handle images).
    Skips EACH output individually if it already exists.
    """

    outputs = {}

    section_type = str(row.get("section_type", "")).strip().upper()
    is_fig = section_type in ["FIG", "FIG_TABLE"]

    print(f"\n🔹 Row {idx} | section_type={section_type}")

    # DeepSeek cannot handle images
    if is_fig:
        print("  ⏭ DeepSeek validation skipped (FIG/Figure type)")
        outputs["deepseekV_claudeE"] = row.get("deepseekV_claudeE")
        outputs["deepseekV_gptE"] = row.get("deepseekV_gptE")
        outputs["deepseekV_gpt4oE"] = row.get("deepseekV_gpt4oE")
        return pd.Series(outputs)

    system_prompt_val = row.get("system_prompt_val")

    # -------------------------
    # DeepSeek on Claude extraction
    # -------------------------
    if pd.notna(row.get("deepseekV_claudeE")):
        print("  ⏭ deepseekV_claudeE already exists. Skipping.")
        outputs["deepseekV_claudeE"] = row.get("deepseekV_claudeE")
    else:
        main_prompt_val_claude = row.get("main_prompt_val_claude")
        if main_prompt_val_claude and system_prompt_val:
            print("  ▶ DeepSeek on Claude extracted CSV...")
            try:
                outputs["deepseekV_claudeE"] = call_deepseek(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_claude,
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["deepseekV_claudeE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["deepseekV_claudeE"] = None

    # -------------------------
    # DeepSeek on GPT extraction
    # -------------------------
    if pd.notna(row.get("deepseekV_gptE")):
        print("  ⏭ deepseekV_gptE already exists. Skipping.")
        outputs["deepseekV_gptE"] = row.get("deepseekV_gptE")
    else:
        main_prompt_val_gpt = row.get("main_prompt_val_gpt")
        if main_prompt_val_gpt and system_prompt_val:
            print("  ▶ DeepSeek on GPT extracted CSV...")
            try:
                outputs["deepseekV_gptE"] = call_deepseek(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_gpt,
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["deepseekV_gptE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["deepseekV_gptE"] = None

    # -------------------------
    # DeepSeek on GPT-4o extraction
    # -------------------------
    if pd.notna(row.get("deepseekV_gpt4oE")):
        print("  ⏭ deepseekV_gpt4oE already exists. Skipping.")
        outputs["deepseekV_gpt4oE"] = row.get("deepseekV_gpt4oE")
    else:
        main_prompt_val_gpt4o = row.get("main_prompt_val_gpt4o")
        if main_prompt_val_gpt4o and system_prompt_val:
            print("  ▶ DeepSeek on GPT4o extracted CSV...")
            try:
                outputs["deepseekV_gpt4oE"] = call_deepseek(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_gpt4o,
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["deepseekV_gpt4oE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["deepseekV_gpt4oE"] = None

    print(f"✅ Row {idx} validation complete")
    return pd.Series(outputs)


In [ ]:
for idx, row in input.iterrows():
    outputs = run_validation_models_deepseek(row, idx)
    for col, val in outputs.items():
        input.at[idx, col] = val

    # Save every 50 rows
    if idx % 50 == 0:
        input.to_csv(f"{output_data_path}/validation_checkpoint_combined.csv", index=False)

# Final save
input.to_csv(f"{output_data_path}/validation_checkpoint_combined.csv", index=False)


In [ ]:
def run_gpt_validation_on_row(row, idx, figures_path):
    """
    Calls GPT on extracted validation prompts for a single row.
    Outputs are saved to new columns: gptV_deepseekE, gptV_claudeE.
    GPT can handle all section types, including FIG/FIG_TABLE.
    If the section is a figure, passes the image path to the model.
    """
    outputs = {}

    section_type = str(row.get("section_type", "")).strip().upper()
    is_fig = section_type in ["FIG", "FIG_TABLE"]

    # Skip if validation outputs already exist
    if pd.notna(row.get("gptV_deepseekE")) and pd.notna(row.get("gptV_claudeE")):
        print(f"Row {idx} ✅ GPT validation outputs already exist. Skipping.")
        return pd.Series(outputs)

    print(f"\n🔹 Row {idx} | section_type={section_type}")

    system_prompt_val = row.get("system_prompt_val")

    # Prepare image path if this is a figure
    image_path = (
        f"{figures_path}/{row['filename']}"
        if is_fig and pd.notna(row.get("filename"))
        else None
    )

    # -------------------------
    # GPT on main_prompt_val_deepseek
    # -------------------------
    main_prompt_val_deepseek = row.get("main_prompt_val_deepseek")
    if main_prompt_val_deepseek and system_prompt_val and not is_fig:
        print("  ▶ GPT on DeepSeek extracted CSV...")
        try:
            outputs["gptV_deepseekE"] = call_gpt(
                system_prompt=system_prompt_val,
                main_prompt=main_prompt_val_deepseek,
                image_path=image_path
            )
            print("    ✓ Done")
        except Exception as e:
            print(f"    ✗ Failed: {e}")
            outputs["gptV_deepseekE"] = None
    else:
        outputs["gptV_deepseekE"] = None
        print("    ⏭ Skipped (no prompt)")

    # -------------------------
    # GPT on main_prompt_val_claude
    # -------------------------
    main_prompt_val_claude = row.get("main_prompt_val_claude")
    if main_prompt_val_claude and system_prompt_val:
        print("  ▶ GPT on Claude extracted CSV...")
        try:
            outputs["gptV_claudeE"] = call_gpt(
                system_prompt=system_prompt_val,
                main_prompt=main_prompt_val_claude,
                image_path=image_path
            )
            print("    ✓ Done")
        except Exception as e:
            print(f"    ✗ Failed: {e}")
            outputs["gptV_claudeE"] = None
    else:
        outputs["gptV_claudeE"] = None
        print("    ⏭ Skipped (no prompt)")

    print(f"✅ Row {idx} GPT validation complete")
    return pd.Series(outputs)


In [ ]:
for idx, row in input.iterrows():
    outputs = run_gpt_validation_on_row(row, idx, figures_path)
    for col, val in outputs.items():
        input.at[idx, col] = val

    # Save every 50 rows
    if idx % 50 == 0:
        input.to_csv(f"{output_data_path}/validation_checkpoint_combined.csv", index=False)

# Final save
input.to_csv(f"{output_data_path}/validation_checkpoint_combined.csv", index=False)


In [ ]:
def run_claude_validation_on_row(row, idx, figures_path):
    """
    Calls Claude on validation prompts for a single row.

    Outputs:
        - claudeV_deepseekE
        - claudeV_gptE
        - claudeV_gpt4oE

    Claude can handle all section types including FIG/FIG_TABLE.
    Skips EACH output individually if it already exists.
    """

    outputs = {}

    section_type = str(row.get("section_type", "")).strip().upper()
    is_fig = section_type in ["FIG", "FIG_TABLE"]

    print(f"\n🔹 Row {idx} | section_type={section_type}")

    system_prompt_val = row.get("system_prompt_val")

    # Prepare image path if figure
    image_path = (
        f"{figures_path}/{row['filename']}"
        if is_fig and pd.notna(row.get("filename"))
        else None
    )

    # -------------------------
    # Claude on DeepSeek extraction
    # -------------------------
    if pd.notna(row.get("claudeV_deepseekE")) and str(row.get("claudeV_deepseekE")).strip() != "":
        print("  ⏭ claudeV_deepseekE already exists. Skipping.")
        outputs["claudeV_deepseekE"] = row.get("claudeV_deepseekE")
    else:
        main_prompt_val_deepseek = row.get("main_prompt_val_deepseek")
        if main_prompt_val_deepseek and system_prompt_val:
            print("  ▶ Claude on DeepSeek extracted CSV...")
            try:
                outputs["claudeV_deepseekE"] = call_claude(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_deepseek,
                    image_path=image_path
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["claudeV_deepseekE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["claudeV_deepseekE"] = None

    # -------------------------
    # Claude on GPT extraction
    # -------------------------
    if pd.notna(row.get("claudeV_gptE")) and str(row.get("claudeV_gptE")).strip() != "":
        print("  ⏭ claudeV_gptE already exists. Skipping.")
        outputs["claudeV_gptE"] = row.get("claudeV_gptE")
    else:
        main_prompt_val_gpt = row.get("main_prompt_val_gpt")
        if main_prompt_val_gpt and system_prompt_val:
            print("  ▶ Claude on GPT extracted CSV...")
            try:
                outputs["claudeV_gptE"] = call_claude(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_gpt,
                    image_path=image_path
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["claudeV_gptE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["claudeV_gptE"] = None

    # -------------------------
    # Claude on GPT4o extraction
    # -------------------------
    if pd.notna(row.get("claudeV_gpt4oE")) and str(row.get("claudeV_gpt4oE")).strip() != "":
        print("  ⏭ claudeV_gpt4oE already exists. Skipping.")
        outputs["claudeV_gpt4oE"] = row.get("claudeV_gpt4oE")
    else:
        main_prompt_val_gpt4o = row.get("main_prompt_val_gpt4o")
        if main_prompt_val_gpt4o and system_prompt_val:
            print("  ▶ Claude on GPT4o extracted CSV...")
            try:
                outputs["claudeV_gpt4oE"] = call_claude(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_gpt4o,
                    image_path=image_path
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["claudeV_gpt4oE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["claudeV_gpt4oE"] = None

    print(f"✅ Row {idx} Claude validation complete")
    return pd.Series(outputs)


In [ ]:
claude_cols = [col for col in input.columns if 'claudeV' in col]
claude_cols


In [ ]:
# Find cells containing error code 400
mask = input[claude_cols].apply(
    lambda col: col.str.contains(
        r'ERROR: Error code: 400\b',
        regex=True,
        na=False
    )
)

# Print indices where at least one Claude column contains the error
error_indices = input.index[mask.any(axis=1)].tolist()

print(f"Replaced {mask.sum().sum()} cells across {len(error_indices)} rows")
print(f"Row indices: {error_indices}")

# Replace entire matching cells with empty strings
input[claude_cols] = input[claude_cols].mask(mask, '')


In [ ]:
for idx, row in input.iterrows():
    outputs = run_claude_validation_on_row(row, idx, figures_path)
    for col, val in outputs.items():
        input.at[idx, col] = val

    # Save every 50 rows
    if idx % 50 == 0:
        input.to_csv(f"{output_data_path}/validation_checkpoint_combined.csv", index=False)

# Final save
input.to_csv(f"{output_data_path}/validation_checkpoint_combined.csv", index=False)


In [ ]:
def run_gpt4o_validation_on_row(row, idx, figures_path):
    """
    Calls GPT on extracted validation prompts for a single row.
    Outputs are saved to new columns: gptV_deepseekE, gptV_claudeE.
    GPT can handle all section types, including FIG/FIG_TABLE.
    If the section is a figure, passes the image path to the model.
    """
    outputs = {}

    section_type = str(row.get("section_type", "")).strip().upper()
    is_fig = section_type in ["FIG", "FIG_TABLE"]

    # Skip if validation outputs already exist
    if pd.notna(row.get("gpt4oV_deepseekE")) and pd.notna(row.get("gpt4oV_claudeE")):
        print(f"Row {idx} ✅ GPT validation outputs already exist. Skipping.")
        return pd.Series(outputs)

    print(f"\n🔹 Row {idx} | section_type={section_type}")

    system_prompt_val = row.get("system_prompt_val")

    # Prepare image path if this is a figure
    image_path = (
        f"{figures_path}/{row['filename']}"
        if is_fig and pd.notna(row.get("filename"))
        else None
    )

    # -------------------------
    # GPT on main_prompt_val_deepseek
    # -------------------------
    main_prompt_val_deepseek = row.get("main_prompt_val_deepseek")
    if main_prompt_val_deepseek and system_prompt_val and not is_fig:
        print("  ▶ GPT4o on DeepSeek extracted CSV...")
        try:
            outputs["gpt4oV_deepseekE"] = call_gpt4o(
                system_prompt=system_prompt_val,
                main_prompt=main_prompt_val_deepseek,
                image_path=image_path
            )
            print("    ✓ Done")
        except Exception as e:
            print(f"    ✗ Failed: {e}")
            outputs["gpt4oV_deepseekE"] = None
    else:
        outputs["gpt4oV_deepseekE"] = None
        print("    ⏭ Skipped (no prompt)")

    # -------------------------
    # GPT on main_prompt_val_claude
    # -------------------------
    main_prompt_val_claude = row.get("main_prompt_val_claude")
    if main_prompt_val_claude and system_prompt_val:
        print("  ▶ GPT4o on Claude extracted CSV...")
        try:
            outputs["gpt4oV_claudeE"] = call_gpt4o(
                system_prompt=system_prompt_val,
                main_prompt=main_prompt_val_claude,
                image_path=image_path
            )
            print("    ✓ Done")
        except Exception as e:
            print(f"    ✗ Failed: {e}")
            outputs["gpt4oV_claudeE"] = None
    else:
        outputs["gpt4oV_claudeE"] = None
        print("    ⏭ Skipped (no prompt)")

    print(f"✅ Row {idx} GPT validation complete")
    return pd.Series(outputs)


In [ ]:
for idx, row in input.iterrows():
    outputs = run_gpt4o_validation_on_row(row, idx, figures_path)
    for col, val in outputs.items():
        input.at[idx, col] = val

    # Save every 50 rows
    if idx % 50 == 0:
        input.to_csv(f"{output_data_path}/validation_checkpoint_combined.csv", index=False)

# Final save
input.to_csv(f"{output_data_path}/validation_checkpoint_combined.csv", index=False)


In [ ]:

# Parse CSV blocks out of the validator outputs (same parser as above).
for col in new_cols:
    new_col = col + "_output"
    print(f"Processing {col} -> {new_col}")
    input[new_col] = input[col].apply(extract_csv)


In [ ]:
input.to_csv(f"{output_data_path}/validation_combined.csv", index=False)


## 7A same model validation 
- read validation_combined.csv and 
- (add columns like claudeV_claudeE, etc and the appropriate functions for validation.

In [ ]:
# ============================================================
# 7A. SAME-MODEL VALIDATION
# ============================================================

import pandas as pd
from pathlib import Path

validation_path = Path(output_data_path) / "validation_combined_new.csv"

# ------------------------------------------------------------
# 1. Load existing validation data
# ------------------------------------------------------------
input = pd.read_csv(validation_path, low_memory=False)

# Same-model validation configurations
SAME_MODEL_CONFIGS = {
    "claudeV_claudeE": {
        "prompt_col": "main_prompt_val_claude",
        "extract_col": "claude_E_output",
        "call": call_claude,
        "supports_figures": True,
    },
    "gptV_gptE": {
        "prompt_col": "main_prompt_val_gpt",
        "extract_col": "gpt_E_output",
        "call": call_gpt,
        "supports_figures": True,
    },
    "gpt4oV_gpt4oE": {
        "prompt_col": "main_prompt_val_gpt4o",
        "extract_col": "gpt4o_E_output",
        "call": call_gpt4o,
        "supports_figures": True,
    },
    "deepseekV_deepseekE": {
        "prompt_col": "main_prompt_val_deepseek",
        "extract_col": "deepseek_E_output",
        "call": call_deepseek,
        "supports_figures": False,
    },
}

# ------------------------------------------------------------
# 2. Add missing columns
# ------------------------------------------------------------
for col in SAME_MODEL_CONFIGS:
    if col not in input.columns:
        input[col] = None

    output_col = f"{col}_output"
    if output_col not in input.columns:
        input[output_col] = None


# ------------------------------------------------------------
# 3. Helpers
# ------------------------------------------------------------
def has_valid_output(value):
    """True if a cell contains a nonempty, non-error response."""
    if value is None:
        return False

    if not isinstance(value, str):
        if pd.isna(value):
            return False
        value = str(value)

    value = value.strip()

    return bool(value) and not value.upper().startswith("ERROR")


def get_same_model_image_path(row):
    """Return the figure image path for FIG/FIG_TABLE rows."""
    section_type = str(row.get("section_type", "")).strip().upper()

    if section_type not in ("FIG", "FIG_TABLE"):
        return None

    filename = row.get("filename")

    if not has_valid_output(filename):
        return None

    return str(Path(figures_path) / str(filename))


# ------------------------------------------------------------
# 4. Run validation for a single row
# ------------------------------------------------------------
def run_same_model_validation_on_row(row, idx):
    outputs = {}

    section_type = str(row.get("section_type", "")).strip().upper()
    is_fig = section_type in ("FIG", "FIG_TABLE")

    system_prompt = row.get("system_prompt_val")
    image_path = get_same_model_image_path(row)

    print(f"\n🔹 Row {idx} | section_type={section_type}")

    for col, config in SAME_MODEL_CONFIGS.items():

        # Skip successful existing outputs
        if has_valid_output(row.get(col)):
            print(f"  ⏭ {col} already exists. Skipping.")
            continue

        # DeepSeek cannot process figures
        if is_fig and not config["supports_figures"]:
            print(f"  ⏭ {col}: DeepSeek figure validation skipped.")
            continue

        # Check extracted relationships exist
        extraction = row.get(config["extract_col"])

        if not has_valid_output(extraction):
            print(f"  ⏭ {col}: No extraction available.")
            continue

        # Get existing validation prompt
        main_prompt = row.get(config["prompt_col"])

        # Rebuild prompt if missing
        if not has_valid_output(main_prompt):
            main_prompt = build_main_prompt_val(
                row,
                config["extract_col"]
            )

        if not has_valid_output(main_prompt) or not has_valid_output(system_prompt):
            print(f"  ⏭ {col}: Missing validation prompt.")
            continue

        # Avoid running visual validation without its image
        if is_fig and not image_path:
            print(f"  ⏭ {col}: Missing figure image path.")
            continue

        if is_fig and not Path(image_path).is_file():
            print(f"  ⏭ {col}: Image file not found: {image_path}")
            continue

        print(f"  ▶ Running {col}...")

        try:
            if config["supports_figures"]:
                result = config["call"](
                    system_prompt=system_prompt,
                    main_prompt=main_prompt,
                    image_path=image_path,
                )
            else:
                result = config["call"](
                    system_prompt=system_prompt,
                    main_prompt=main_prompt,
                )

            if has_valid_output(result):
                outputs[col] = result
                print("    ✓ Done")
            else:
                outputs[col] = result
                print("    ✗ Empty or error response")

        except Exception as e:
            outputs[col] = f"ERROR: {e}"
            print(f"    ✗ Failed: {e}")

    return outputs


# ------------------------------------------------------------
# 5. Run across all rows
# ------------------------------------------------------------
SAVE_EVERY = 50

for position, (idx, row) in enumerate(input.iterrows(), start=1):

    outputs = run_same_model_validation_on_row(row, idx)

    for col, val in outputs.items():
        input.at[idx, col] = val

        # Parse validation CSV output
        output_col = f"{col}_output"

        if has_valid_output(val):
            input.at[idx, output_col] = extract_csv(val)
        else:
            input.at[idx, output_col] = None

    # Checkpoint every 50 rows
    if position % SAVE_EVERY == 0:
        input.to_csv(validation_path, index=False)
        print(f"\n💾 Saved checkpoint: {position}/{len(input)} rows")

# ------------------------------------------------------------
# 6. Parse all same-model validation outputs
# ------------------------------------------------------------
for col in SAME_MODEL_CONFIGS:
    output_col = f"{col}_output"

    input[output_col] = input[col].apply(
        lambda x: extract_csv(x) if has_valid_output(x) else None
    )

# ------------------------------------------------------------
# 7. Final save
# ------------------------------------------------------------
input.to_csv(validation_path, index=False)

print("\n✅ Same-model validation complete!")
print(f"Saved to: {validation_path}")

# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------
print("\nSame-model validation summary:")

for col in SAME_MODEL_CONFIGS:
    raw_count = input[col].apply(has_valid_output).sum()
    parsed_count = input[f"{col}_output"].notna().sum()

    print(
        f"{col}: "
        f"{raw_count} raw outputs | "
        f"{parsed_count} parsed CSV outputs"
    )

## 8. Refinement: CTD name and ID normalization


In [ ]:

ctd_chemicals = pd.read_csv(f"{ctd_data_path}/CTD_chemicals.csv", skiprows=27)
ctd_genes = pd.read_csv(f"{ctd_data_path}/CTD_genes.csv", skiprows=27)


In [ ]:
def repair_csv(val_str: str) -> str:
    """
    Repairs CSV strings where a free-text field (e.g. geneName) may contain
    commas, causing row field counts to exceed the header field count.

    Strategy: Instead of hardcoding column positions, this function:
      1. Parses the header to get the expected column count (N).
      2. For each malformed row (too many fields), finds the LONGEST
         contiguous run of 'ambiguous' tokens — i.e. the tokens that,
         when collapsed back into one comma-joined string, produce exactly
         N fields total.
      3. Identifies which column is the free-text one by scanning the header
         for known free-text field names (e.g. 'geneName', 'explanation').
         Falls back to trying every possible merge window if no known field found.

    This means it works correctly whether the schema is:
        chemicalName, chemicalID, relationship, geneName, geneID          (5 cols)
        chemicalName, relationship, geneName, geneSymbol, agreement, reasoning  (6 cols)
        ...or any other schema where exactly one field may contain commas.
    """

    # Known free-text fields that may contain commas, in priority order
    FREE_TEXT_FIELDS = [
        'genename', 'explanation', 'reasoning', 'description',
        'caption', 'text', 'name'
    ]

    # ✅ Extract CSV blocks from markdown code fences
    # If there are multiple ```csv blocks, keep only the last one
    if '```' in val_str:
        blocks = []
        current_block = []
        in_code_block = False
        
        for line in val_str.splitlines():
            stripped = line.strip()
            if stripped.startswith('```'):
                if in_code_block:
                    # End of block
                    if current_block:
                        blocks.append('\n'.join(current_block))
                    current_block = []
                    in_code_block = False
                else:
                    # Start of block
                    in_code_block = True
            elif in_code_block:
                current_block.append(line)
        
        # If there are blocks, use the last one; otherwise use original
        if blocks:
            val_str = blocks[-1].strip()

    lines = val_str.strip().splitlines()
    if not lines:
        return val_str

    # Parse header
    try:
        header_parts = next(csv.reader([lines[0]]))
    except Exception:
        return val_str

    expected = len(header_parts)

    # Find which column index is the free-text field
    free_text_idx = None
    header_lower = [h.strip().lower() for h in header_parts]
    for field in FREE_TEXT_FIELDS:
        if field in header_lower:
            free_text_idx = header_lower.index(field)
            break

    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_MINIMAL)
    writer.writerow(header_parts)

    for line in lines[1:]:
        if not line.strip():
            continue

        # Parse the line respecting existing quotes first
        try:
            parts = next(csv.reader([line]))
        except Exception:
            # If quoted parsing fails, fall back to raw split
            parts = line.split(',')

        if len(parts) == expected:
            writer.writerow(parts)
            continue

        if len(parts) < expected:
            # Under-count — write as-is, let pandas handle/warn
            writer.writerow(parts)
            continue

        # Too many fields — need to merge some tokens
        extra = len(parts) - expected  # how many extra commas exist

        if free_text_idx is not None:
            # We know which field to merge into: collapse `extra+1` tokens
            # starting at free_text_idx into a single value
            merge_count = extra + 1
            prefix = parts[:free_text_idx]
            merged = ','.join(parts[free_text_idx: free_text_idx + merge_count])
            suffix = parts[free_text_idx + merge_count:]
            repaired = prefix + [merged] + suffix
            if len(repaired) == expected:
                writer.writerow(repaired)
                continue

        # Fallback: try every possible merge window and pick the first
        # one that produces exactly `expected` fields
        repaired = None
        for start in range(len(parts) - extra):
            end = start + extra + 1  # merge tokens[start:end]
            candidate = parts[:start] + [','.join(parts[start:end])] + parts[end:]
            if len(candidate) == expected:
                repaired = candidate
                break

        if repaired:
            writer.writerow(repaired)
        else:
            # Can't repair — write raw and let pandas warn
            writer.writerow(parts)

    return output.getvalue()


# ── Rest of the pipeline (unchanged except repair_csv is now called) ──────────

output_cols = [
    # Extraction
    'deepseek_E_output',
    'gpt_E_output',
    'claude_E_output',
    'gpt4o_E_output',

    # DeepSeek validation
    'deepseekV_gptE_output',
    'deepseekV_gpt4oE_output',
    'deepseekV_claudeE_output',
    'deepseekV_deepseekE_output',

    # GPT validation
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    'gptV_gptE_output',

    # Claude validation
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'claudeV_gpt4oE_output',
    'claudeV_claudeE_output',

    # GPT-4o validation
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output',
    'gpt4oV_gpt4oE_output',
]

keep_cols = [
    'explanation',
    'chemicalName',
    'chemicalID',
    'relationship',
    'geneName',
    'geneID',
    'geneSymbol',
    'source',
    '_source_row',
    'agreement',
    'reasoning'
    # 'confidence'
]


def combine_csv_outputs(df, columns):
    all_dfs = {}
    extra_cols = [
        'pmid',
        'caption',
        'original_filename',
        'gene_map',
        'chemical_map',
        'section_type',
        'text',
        'table_xml'
    ]

    for col in columns:
        if col not in df.columns:
            continue

        col_dfs = []
        for idx, val in df[col].items():
            if val is None:
                continue
            if isinstance(val, pd.DataFrame):
                col_dfs.append(val)
                continue

            try:
                val_str = str(val).strip()
            except Exception:
                continue
            if not val_str:
                continue

            # ✅ Repair comma-in-free-text-field issues AND extract from code fences
            val_str = repair_csv(val_str)

            try:
                temp_df = pd.read_csv(
                    io.StringIO(val_str),
                    engine="python",
                    sep=",",
                    quotechar='"',
                    skipinitialspace=True,
                    on_bad_lines="warn"
                )
                temp_df["_source_row"] = idx
                for c in extra_cols:
                    temp_df[c] = df.at[idx, c] if c in df.columns else pd.NA
                col_dfs.append(temp_df)
            except Exception as e:
                print(f"⚠️ Failed parsing row {idx} column {col}: {e}")
                continue

        if col_dfs:
            combined_col_df = pd.concat(col_dfs, ignore_index=True, sort=False)
            for c in keep_cols:
                if c not in combined_col_df.columns:
                    combined_col_df[c] = pd.NA
            final_cols = ['_source_row'] + extra_cols + keep_cols
            combined_col_df = combined_col_df[[c for c in final_cols if c in combined_col_df.columns]]
            all_dfs[col] = combined_col_df
            print(f"done with {col} - total rows = {len(combined_col_df)}")
        else:
            print(f"done with {col} - no valid CSV")

    return all_dfs


In [ ]:
def build_ctd_chemical_lookup(ctd_chemicals_df):
    chem_id_to_info = {}
    name_to_chem_id = {}

    # Make column names consistent
    ctd_chemicals_df = ctd_chemicals_df.rename(columns=lambda x: x.strip())

    for _, row in ctd_chemicals_df.iterrows():
        chem_name = str(row.get("# ChemicalName", "") or row.get("Chemical Name", "")).strip()
        chem_id = str(row.get("ChemicalID", "") or row.get("Chemical ID", "")).replace("MESH:", "").upper()
        
        # synonyms
        mesh_syn = str(row.get("MESHSynonyms", "")).split("|") if row.get("MESHSynonyms") else []
        ctd_syn = str(row.get("CTDCuratedSynonyms", "")).split("|") if row.get("CTDCuratedSynonyms") else []
        synonyms = [s.strip() for s in mesh_syn + ctd_syn if s.strip()]

        chem_id_to_info[chem_id] = {"chem_name": chem_name, "synonyms": synonyms}

        if chem_name:
            name_to_chem_id[chem_name.upper()] = chem_id

        for syn in synonyms:
            name_to_chem_id[syn.upper()] = chem_id

    return chem_id_to_info, name_to_chem_id



def normalize_gene_id(x):
    """Convert GeneID to clean string or None"""
    if pd.isna(x):
        return None
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return str(x).strip()

def build_ctd_gene_lookups(ctd_genes_df):
    """
    Build lookup dictionaries from CTD genes DataFrame.
    
    Returns:
      geneid_to_name:   {GeneID -> canonical GeneName}
      geneid_to_symbol:{GeneID -> GeneSymbol}
      name_to_geneid:  {UPPERCASE name/symbol/synonym -> GeneID}
      alt_to_geneid:   {AltGeneID -> GeneID}
    """
    geneid_to_name = {}
    geneid_to_symbol = {}
    name_to_geneid = {}
    alt_to_geneid = {}

    for _, row in ctd_genes_df.iterrows():
        gene_id = normalize_gene_id(row.get("GeneID"))
        if not gene_id:
            continue

        symbol = row.get("# GeneSymbol")
        # print(symbol)
        name = row.get("GeneName")
        synonyms = row.get("Synonyms")
        alt_ids = row.get("AltGeneIDs")

        # Canonical name: prefer GeneName over symbol
        canonical_name = name if isinstance(name, str) and name.strip() else symbol
        if not isinstance(canonical_name, str) or not canonical_name.strip():
            continue

        geneid_to_name[gene_id] = canonical_name
        geneid_to_symbol[gene_id] = symbol.strip() if isinstance(symbol, str) else None

        # Index symbol + name
        for n in [symbol, name]:
            if isinstance(n, str) and n.strip():
                name_to_geneid[n.upper()] = gene_id

        # Index synonyms
        if isinstance(synonyms, str) and synonyms.strip():
            for syn in synonyms.split("|"):
                syn = syn.strip()
                if syn:
                    name_to_geneid[syn.upper()] = gene_id

        # Index alternate GeneIDs
        if isinstance(alt_ids, str) and alt_ids.strip():
            for alt in alt_ids.split("|"):
                alt = normalize_gene_id(alt)
                if alt:
                    alt_to_geneid[alt] = gene_id

    return geneid_to_name, geneid_to_symbol, name_to_geneid, alt_to_geneid


def update_chemical(df, chem_id_to_info, name_to_chem_id, verbose=True):
    """
    Update chemicalName and chemicalID efficiently using prebuilt lookup tables.
    Flags:
        chemicalFlag = 0 : found normally
        chemicalFlag = 1 : not found
    """
    df = df.copy()
    df['chemicalFlag'] = 0

    # Normalize chemicalID column
    df['chemicalID'] = df['chemicalID'].apply(
        lambda x: str(x).upper().replace("MESH:", "").strip() if pd.notna(x) else None
    )
    df['chemicalName'] = df['chemicalName'].fillna("").astype(str)

    # Precompute normalized name_to_chem_id dict
    norm_name_to_id = {
        k.upper().replace("-", "").replace(" ", ""): v
        for k, v in name_to_chem_id.items()
    }

    for i, row in df.iterrows():
        chem_orig = row['chemicalName']
        chemicalName = chem_orig.strip()
        chemicalID = row['chemicalID']
        chemical_found = False

        # 1️⃣ Exact match by chemicalID
        if chemicalID and chemicalID in chem_id_to_info:
            df.at[i, 'chemicalName'] = chem_id_to_info[chemicalID]['chem_name']
            if verbose:
                print(f"Row {i} - Exact ID match: {chem_id_to_info[chemicalID]['chem_name']}")
            chemical_found = True
            continue  # already found

        # 2️⃣ Exact match by normalized chemicalName
        query_upper = chemicalName.upper().replace("-", "").replace(" ", "")
        if query_upper in norm_name_to_id:
            matched_id = norm_name_to_id[query_upper]
            df.at[i, 'chemicalID'] = matched_id
            df.at[i, 'chemicalName'] = chem_id_to_info[matched_id]['chem_name']
            if verbose:
                print(f"Row {i} - Exact name match: {chem_id_to_info[matched_id]['chem_name']}")
            chemical_found = True
            continue

        # 3️⃣ Partial match
        for key_norm, cid in norm_name_to_id.items():
            if query_upper and query_upper in key_norm:
                df.at[i, 'chemicalID'] = cid
                df.at[i, 'chemicalName'] = chem_id_to_info[cid]['chem_name']
                chemical_found = True
                if verbose:
                    print(f"Row {i} - Partial match: {chem_id_to_info[cid]['chem_name']}")
                break

        # 4️⃣ If still not found
        if not chemical_found:
            df.at[i, 'chemicalFlag'] = 1
            if not df.at[i, 'chemicalName']:
                df.at[i, 'chemicalName'] = chem_orig
            if verbose:
                print(f"Row {i} - Not found in CTD chemicals")
                print(f"  chemicalName: {chem_orig}")
                print(f"  chemicalID:   {chemicalID}")
                print("-" * 40)

    return df

def update_gene(df, geneid_to_name, geneid_to_symbol, name_to_geneid, alt_to_geneid, verbose=True):
    """
    Update geneName, geneID, and geneSymbol in the DataFrame using CTD gene lookups.

    Flags:
        geneFlag = 0 : found normally
        geneFlag = 1 : not found

    Additionally:
        oldGeneID = original geneID before normalization/update
    """
    df = df.copy()
    df["geneFlag"] = 0
    
    # ✅ Store original geneID before any modification
    df["oldGeneID"] = df["geneID"]
    df["oldGeneName"] = df["geneName"]
    df["oldGeneSymbol"] = df["geneSymbol"]

    for i, row in df.iterrows():
        original_geneID = row.get("geneID")
        geneID = normalize_gene_id(original_geneID)
        geneName = row.get("geneName")
        geneSymbol = row.get("geneSymbol", None)

        found = False

        # 1️⃣ Resolve by GeneID
        if geneID:
            if geneID in geneid_to_name:
                geneName = geneid_to_name[geneID]
                geneSymbol = geneid_to_symbol.get(geneID)
                found = True

            elif geneID in alt_to_geneid:
                parent_id = alt_to_geneid[geneID]
                geneID = parent_id
                geneName = geneid_to_name.get(parent_id)
                geneSymbol = geneid_to_symbol.get(parent_id)
                found = True
        # somewhere in here, geneSymbol is getting overwritten to geneName.
        
        # 2️⃣ Flags
        if not found:
            df.at[i, "geneFlag"] = 1
            if verbose:
                print(f"Row {i} not found in CTD genes")
                print(f"  geneName: {geneName}")
                print(f"  geneID:   {geneID}")
                print("-" * 40)
        else:
            df.at[i, "geneFlag"] = 0

        # ✅ Overwrite geneID (while oldGeneID is preserved)
        df.at[i, "geneName"] = geneName
        df.at[i, "geneID"] = geneID
        df.at[i, "geneSymbol"] = geneSymbol

    return df


In [ ]:
def update_relationship(df, verbose=True):
    # Allowed CTD relationships
    relationships = [
        "cotreatment", "binding", "reaction", "activity", "localization", "expression", "abundance", 
        "mutagenesis", "response to chemical", "stability", "splicing", "folding", "transport", 
        "uptake", "import", "secretion", "export", "metabolic processing", "chemical synthesis", 
        "degradation", "acetylation", "acylation", "alkylation", "amination", "carbamoylation", 
        "carboxylation", "cleavage", "ethylation", "glycation", "glycosylation", "N-linked glycosylation", 
        "O-linked glycosylation", "glucuronidation", "hydrolysis", "hydroxylation", "lipidation", 
        "geranoylation", "farnesylation", "myristoylation", "palmitoylation", "prenylation", 
        "methylation", "nitrosation", "nucleotidylation", "oxidation", "phosphorylation", "sulfation", 
        "reduction", "ribosylation", "ADP-ribosylation", "ubiquitination", "sumoylation", 
        "glutathionylation", "polymerization"
    ]
    lower_relationships = {r.lower() for r in relationships}

    # Prefix mapping: canonical form
    prefix_map = {
        'increase': 'increases', 'increases': 'increases', 'increased': 'increases', 'increasing': 'increases',
        'decrease': 'decreases', 'decreases': 'decreases', 'decreased': 'decreases', 'decreasing': 'decreases',
        'affect': 'affects', 'affects': 'affects', 'affected': 'affects', 'affecting': 'affects'
    }

    # Prefix exceptions: always "affects"
    affects_required = ['binding', 'cotreatment']

    # Chemical-chemical only relationships
    chem_chem_only = ['abundance', 'response to chemical', 'chemical synthesis']

    # Inhibitory terms replaced with "decrease"
    inhibit_terms = ['inhibit', 'inhibits', 'inhibited', 'inhibiting', 'inhibition']

    df = df.copy()
    df['relationshipWarningFlag'] = 0
    df['prefixMissingFlag'] = 0  # NEW: initialize flag

    if 'relationship' not in df.columns:
        df['relationship'] = ''

    # Normalize text: lowercase, remove quotes, strip spaces
    rel_series = df['relationship'].astype(str).str.lower().str.replace('"', '').str.strip()
    rel_series = rel_series.apply(lambda x: re.sub(r'\s+', ' ', x))

    # Replace inhibitory terms with "decrease"
    for term in inhibit_terms:
        rel_series = rel_series.str.replace(rf'\b{term}\b', 'decrease', regex=True)

    # Normalize prefix and apply rules
    def normalize(rel):
        words = rel.split()
        prefix = words[0] if words and words[0] in prefix_map else None
        core = ' '.join(words[1:]) if prefix else rel

        # NEW: set prefixMissingFlag if no prefix
        if not prefix:
            df.at[df.index[rel_series[rel_series == rel].index[0]], 'prefixMissingFlag'] = 1

        # Prefix exceptions
        if core in affects_required:
            return f"affects {core}"

        # Apply canonical prefix if prefix exists
        if prefix:
            new_prefix = prefix_map.get(prefix, prefix)
            return f"{new_prefix} {core}" if core else new_prefix

        return core

    rel_series = rel_series.apply(normalize)

    # Flag prefix and relationship validity (single pass)
    def flag_prefix(rel):

        words = rel.split()
        prefix = words[0] if words else None
        valid_prefixes = {'increase', 'increases', 'decrease', 'decreases', 'affect', 'affects'}
    
        # Flag if no valid prefix
        if prefix not in valid_prefixes:
            return 1
    
        core = ' '.join(words[1:])
    
        # chem-chem only relationships always invalid
        if core in chem_chem_only:
            return 1
    
        # Invalid if core not in allowed relationships
        return 0 if core in lower_relationships else 1

    df['relationship'] = rel_series
    df['relationshipWarningFlag'] = rel_series.apply(flag_prefix)

    if verbose:
        invalid_df = df[df['relationshipWarningFlag'] == 1]
        n_invalid = len(invalid_df)
        print(f"{n_invalid} invalid relationships detected.")
        if n_invalid > 0:
            cols_to_show = ['relationship', 'relationshipWarningFlag', 'prefixMissingFlag']
            for col in ['chemicalID', 'geneID', 'chemicalFlag', 'geneFlag']:
                if col in df.columns:
                    cols_to_show.insert(0, col)
            print(invalid_df[cols_to_show].head(20))

    return df


In [ ]:


# ---------------------------------------------------------------------------
# Small pure helpers
# ---------------------------------------------------------------------------

def _symbol_numeric_suffix(symbol) -> Optional[str]:
    """
    Extract the portion of a GeneSymbol starting from its first digit.
        FKBP1A -> "1A"
        IL6    -> "6"
        TP53   -> "53"
    Returns None if no digit is found or input is NaN.
    """
    if pd.isna(symbol):
        return None
    m = re.search(r"\d.*", str(symbol))
    return m.group(0) if m else None


def _tokenize(name) -> list:
    """Split a (already-cleaned) name string into lowercase tokens."""
    if pd.isna(name) or not str(name).strip():
        return []
    return str(name).strip().lower().split()


def _remove_parentheses(name) -> str:
    """
    Strip parenthesised substrings (including the parentheses themselves).
        "glyceraldehyde-3-phosphate dehydrogenase (GAPDH)"
        -> "glyceraldehyde-3-phosphate dehydrogenase"
    """
    if pd.isna(name):
        return ""
    return re.sub(r"\s*\(.*?\)", "", str(name)).strip()


def _suffix_in_tokens(suffix: Optional[str], tokens: list) -> bool:
    """
    Return True when *suffix* (e.g. "53") is represented in *tokens*.

    Single-token names  : suffix must appear *anywhere* in the token.
    Multi-token names   : at least one token must *end with* suffix.
    """
    if not suffix:
        return False
    suffix = suffix.lower()
    if len(tokens) == 1:
        return suffix in tokens[0]
    return any(tok.endswith(suffix) for tok in tokens)


def _normalize(s) -> str:
    """Lowercase, strip, coerce NaN -> empty string."""
    if pd.isna(s):
        return ""
    return str(s).strip().lower()


def _alt_count(alt_ids_str) -> int:
    """Count non-empty pipe-separated alternative gene IDs."""
    return len([a for a in str(alt_ids_str).split("|") if a.strip()])


def _symbol_is_missing(symbol) -> bool:
    """Return True if a geneSymbol value should be treated as absent."""
    if pd.isna(symbol):
        return True
    return str(symbol).strip() in ("", "-")


def _strip_dot_suffix(symbol) -> Optional[str]:
    """
    If a symbol contains a dot, return the portion before it.
        BRCA1.2  -> "BRCA1"
        TP53.1   -> "TP53"
        BRCA1    -> None  (no dot present)
    """
    if pd.isna(symbol):
        return None
    s = str(symbol).strip()
    if "." in s:
        return s.split(".")[0]
    return None


# ---------------------------------------------------------------------------
# CTD pre-processing  (call once, reuse across batches)
# ---------------------------------------------------------------------------

def prepare_ctd(ctd_genes: pd.DataFrame) -> dict:
    """
    Build fast lookup structures from the CTD gene table.

    Returns a dict with:
        "df"           : cleaned CTD dataframe with helper columns
        "id_index"     : {gene_id_str          -> list[row_index]}
        "symbol_index" : {normalised_symbol    -> list[row_index]}
        "name_index"   : {normalised_gene_name -> list[row_index]}
        "synonym_index": {normalised_synonym   -> list[row_index]}
    """
    ctd = ctd_genes.copy()
    ctd["GeneID"]       = ctd["GeneID"].astype(str)
    ctd["_symbol_norm"] = ctd["# GeneSymbol"].apply(_normalize)
    ctd["_name_norm"]   = ctd["GeneName"].apply(_normalize)
    ctd["_alt_count"]   = ctd["AltGeneIDs"].fillna("").apply(_alt_count)
    ctd["_syns"]        = ctd["Synonyms"].fillna("").str.lower().str.split("|")

    id_index: dict = {}
    for i, gid in ctd["GeneID"].items():
        if gid:
            id_index.setdefault(gid, []).append(i)

    symbol_index: dict = {}
    for i, sym in ctd["_symbol_norm"].items():
        if sym:
            symbol_index.setdefault(sym, []).append(i)

    name_index: dict = {}
    for i, name in ctd["_name_norm"].items():
        if name:
            name_index.setdefault(name, []).append(i)

    synonym_index: dict = {}
    for i, syns in ctd["_syns"].items():
        for s in syns:
            s = s.strip()
            if s:
                synonym_index.setdefault(s, []).append(i)

    return {
        "df": ctd,
        "id_index": id_index,
        "symbol_index": symbol_index,
        "name_index": name_index,
        "synonym_index": synonym_index,
    }


# ---------------------------------------------------------------------------
# Per-row resolution helpers
# ---------------------------------------------------------------------------

def _resolve_by_id(gene_id: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 1: exact GeneID match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["id_index"].get(gene_id, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_symbol(symbol_norm: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 2: exact normalised GeneSymbol match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["symbol_index"].get(symbol_norm, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_name(name_norm: str, name_tokens: list, ctd_lookup: dict) -> Optional[pd.Series]:
    """
    Priority 3: match via GeneName or Synonyms with alt_count tiebreaking.

    Cases
    -----
    1. Only GeneName hits  : prefer suffix-matching candidates; tiebreak by alt_count
    2. Both hits           : compare total alt_count of each group; pick winner's best
    3. Only Synonym hits   : pick highest alt_count
    """
    if not name_norm:
        return None

    ctd_df        = ctd_lookup["df"]
    name_index    = ctd_lookup["name_index"]
    synonym_index = ctd_lookup["synonym_index"]

    gn_rows  = ctd_df.loc[name_index.get(name_norm, [])]
    syn_rows = ctd_df.loc[synonym_index.get(name_norm, [])]

    # Deduplicate: remove synonym hits already present in gene-name hits
    if not gn_rows.empty and not syn_rows.empty:
        syn_rows = syn_rows[~syn_rows.index.isin(gn_rows.index)]

    # ---- Case 1: only GeneName hits ----
    if not gn_rows.empty and syn_rows.empty:
        suffix_matches = [
            r for _, r in gn_rows.iterrows()
            if _suffix_in_tokens(_symbol_numeric_suffix(r["# GeneSymbol"]), name_tokens)
        ]
        best_df = pd.DataFrame(suffix_matches) if suffix_matches else gn_rows
        return best_df.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 2: both hits ----
    if not gn_rows.empty and not syn_rows.empty:
        if gn_rows["_alt_count"].sum() >= syn_rows["_alt_count"].sum():
            return gn_rows.sort_values("_alt_count", ascending=False).iloc[0]
        else:
            return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 3: only Synonym hits ----
    if not syn_rows.empty:
        return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    return None


# ---------------------------------------------------------------------------
# Dot-symbol resolution helper
# ---------------------------------------------------------------------------

def _apply_dot_check(best: pd.Series, ctd_lookup: dict) -> pd.Series:
    """
    If the resolved symbol contains a dot (e.g. BRCA1.2), strip the suffix
    and re-look up the base symbol in CTD to get the correct GeneID/GeneName.
    Returns the corrected row, or the original row with the dot stripped if
    no CTD entry is found for the base.
    """
    base = _strip_dot_suffix(best["# GeneSymbol"])
    if base is None:
        return best  # no dot, nothing to do

    hit = _resolve_by_symbol(_normalize(base), ctd_lookup)
    if hit is not None:
        return hit

    # No CTD entry for base symbol — strip the dot suffix in-place
    best = best.copy()
    best["# GeneSymbol"] = base
    return best


# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------
def update_gene_symbol_(
    df: pd.DataFrame,
    ctd_genes: pd.DataFrame,
    ctd_lookup: Optional[dict] = None,
) -> pd.DataFrame:

    df = df.copy()

    df["oldGeneID"]     = df.get("geneID")
    df["oldGeneName"]   = df.get("geneName")
    df["oldGeneSymbol"] = df.get("geneSymbol")

    # ✅ Treat "-", " -", "- " etc. as missing across key gene columns
    for col in ["geneID", "geneSymbol", "geneName"]:
        if col in df.columns:
            df[col] = df[col].replace(r"^\s*-\s*$", "", regex=True).fillna("")

    if ctd_lookup is None:
        ctd_lookup = prepare_ctd(ctd_genes)

    # Vectorised pre-normalisation
    df["_name_norm"] = (
        df.get("geneName", pd.Series(dtype=str))
        .apply(_remove_parentheses)
        .apply(_normalize)
    )
    df["_symbol_norm"] = (
        df.get("geneSymbol", pd.Series(dtype=str))
        .apply(_normalize)
    )

    flags       = np.full(len(df), 2, dtype=int)
    new_symbols = df["geneSymbol"].copy() if "geneSymbol" in df.columns else pd.Series(index=df.index, dtype=str)
    new_ids     = df["geneID"].astype(str).copy()
    new_names   = df["geneName"].copy()   if "geneName"   in df.columns else pd.Series(index=df.index, dtype=str)

    for pos, (idx, row) in enumerate(df.iterrows()):
        raw_symbol  = row.get("geneSymbol")
        gene_id     = str(row.get("geneID", "")).strip()
        symbol_norm = row["_symbol_norm"]
        name_norm   = row["_name_norm"]
        name_tokens = _tokenize(name_norm)

        # Already valid symbol
        if not _symbol_is_missing(raw_symbol):
            if not gene_id:  # symbol present but ID missing — try to resolve ID from symbol
                best = _resolve_by_symbol(symbol_norm, ctd_lookup)
                if best is not None:
                    best = _apply_dot_check(best, ctd_lookup)
                    new_symbols.iloc[pos] = best["# GeneSymbol"]
                    new_ids.iloc[pos]     = best["GeneID"]
                    new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue
    
        # Priority 1: GeneID
        if gene_id:
            best = _resolve_by_id(gene_id, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 2: GeneSymbol
        if symbol_norm:
            best = _resolve_by_symbol(symbol_norm, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 3a: treat geneName as a symbol
        best = _resolve_by_symbol(name_norm, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue

        # Priority 3b: GeneName / Synonym
        best = _resolve_by_name(name_norm, name_tokens, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0

    df["geneSymbol"]     = new_symbols.values
    df["geneID"]         = new_ids.values
    df["geneName"]       = new_names.values
    df["geneSymbolFlag"] = flags

    df.drop(columns=["_name_norm", "_symbol_norm"], inplace=True)
    return df


In [ ]:
chem_id_to_info, name_to_chem_id = build_ctd_chemical_lookup(ctd_chemicals)
geneid_to_name, geneid_to_symbol, name_to_geneid, alt_to_geneid = build_ctd_gene_lookups(ctd_genes)


In [ ]:
def refine(df, ctd_chemicals, ctd_genes):
    updated_df = update_chemical(
        df=df, 
        chem_id_to_info=chem_id_to_info, 
        name_to_chem_id=name_to_chem_id,
        verbose=False
    )

    updated_df = update_gene(
        updated_df,
        geneid_to_name,
        geneid_to_symbol,
        name_to_geneid,
        alt_to_geneid,
        verbose=False
    )

    # updated_df = update_gene_symbol_(
    #     updated_df,
    #     ctd_genes
    # )

    # call again to check if gene is in CTD or not
    # updated_df = check_gene_flag(updated_df, geneid_to_name, alt_to_geneid)

    updated_df = update_relationship(updated_df, verbose = False)
    return updated_df


In [ ]:
output_cols = [
    # Extraction
    'deepseek_E_output',
    'gpt_E_output',
    'claude_E_output',
    'gpt4o_E_output',

    # DeepSeek validation
    'deepseekV_gptE_output',
    'deepseekV_gpt4oE_output',
    'deepseekV_claudeE_output',
    'deepseekV_deepseekE_output',

    # GPT validation
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    'gptV_gptE_output',

    # Claude validation
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'claudeV_gpt4oE_output',
    'claudeV_claudeE_output',

    # GPT-4o validation
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output',
    'gpt4oV_gpt4oE_output',
]

data = pd.read_csv(f"{output_data_path}/validation_combined.csv")
data.columns

combined_df = combine_csv_outputs(data, output_cols)

os.makedirs(f"{output_data_path}/refined_combined", exist_ok=True)

for col in output_cols:
    output_file = f'{output_data_path}/refined_combined/{col}.csv'

    if os.path.exists(output_file):
        print(f'Skipping {col}, file already exists.')
        continue

    print(f'Starting: {col}')
    temp_df = refine(combined_df[col], ctd_chemicals, ctd_genes)
    temp_df.to_csv(output_file, index=False)
    print(f'Finished: {col}')